# Automated Legal Gap Analysis — GCC Personal Data Protection Laws vs. GDPR

Provision-level comparison of six GCC data protection statutes against the GDPR,
performed by a metacognitively regulated multi-agent system.

**Solver** `gpt-5` · **Critic** `meta-llama/Llama-3.3-70B-Instruct-Turbo` ·
**Runtime** Colab Pro+ / T4 (GPU used for embeddings only)

| Property | Mechanism |
|---|---|
| Reproducibility | Global seed · frozen config hash · full JSONL trace of every model call · deterministic state-machine routing |
| Leakage control | Reference-unique n-grams fingerprinted and blocked from every agent prompt; a match raises |
| Resumability | Append-only checkpoints keyed by record id; re-running skips completed work |
| Throughput | Bounded-concurrency async pool with retry and periodic flush |

Cells run top to bottom. Any cell may be re-run safely.

## 1 · Environment

In [ ]:
# Dependencies
!pip -q install "openai>=1.40" "together>=1.3" pymupdf pdf2image pytesseract sentence-transformers faiss-cpu krippendorff statsmodels matplotlib seaborn pandas numpy scipy scikit-learn tqdm nest_asyncio orjson
!apt-get -qq update
!apt-get -qq install -y poppler-utils tesseract-ocr tesseract-ocr-ara

import shutil, subprocess
print('\n--- binary check ---')
print('pdftotext :', shutil.which('pdftotext') or 'MISSING (PyMuPDF fallback will be used)')
print('tesseract :', shutil.which('tesseract') or 'MISSING')
try:
    _r = subprocess.run(['tesseract','--list-langs'], capture_output=True)
    _out = (_r.stdout + _r.stderr).decode()
    print('tesseract languages:', [l for l in _out.split() if len(l) == 3])
except Exception as _e:
    print('tesseract languages: unavailable', _e)

In [ ]:
import os, re, io, json, math, time, hashlib, random, asyncio, warnings, itertools, shutil, sys
from dataclasses import dataclass, asdict, field
from typing import List, Dict, Any, Optional, Tuple
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import nest_asyncio; nest_asyncio.apply()
warnings.filterwarnings("ignore")

# ---------------- reproducibility ----------------
SEED = 20260801
random.seed(SEED); np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
try:
    import torch
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("torch", torch.__version__, "| device:", DEVICE,
          "|", torch.cuda.get_device_name(0) if DEVICE=="cuda" else "")
except Exception as e:
    DEVICE = "cpu"; print("torch unavailable:", e)

sns.set_theme(style="whitegrid")
KAU_GREEN, KAU_GOLD = "#1F5C3A", "#B8912F"
PALETTE = [KAU_GREEN, KAU_GOLD, "#4E8098", "#A44A3F", "#7D8CA3", "#5C7457"]

## 2 · Paths and run manifest

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive")
DATA_ROOT  = DRIVE_ROOT / "EXP-1"          # main data folder
if not DATA_ROOT.exists():
    here = [q.name for q in DRIVE_ROOT.iterdir()][:40]
    raise FileNotFoundError(
        f"Not found: {DATA_ROOT}\nTop of My Drive: {here}\n"
        "If EXP-1 sits in 'Shared with me', add a shortcut to My Drive first.")
print("DATA_ROOT →", DATA_ROOT)
print("PDFs found:", len(list(DATA_ROOT.glob("*.pdf"))))

RUN_TAG   = os.environ.get("RUN_TAG", "run-001")          # bump to start a fresh run
OUT       = DATA_ROOT / f"OUTPUT_{RUN_TAG}"
DIRS = {k: OUT/k for k in
        ["00_manifest","01_corpus","02_requirement_units","03_reference_kb","04_index",
         "05_traces","06_records","07_perturbations","08_ablation","09_metrics",
         "10_figures","11_report"]}
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)
print("Outputs →", OUT)

# ---------------- frozen configuration ----------------
CONFIG = dict(
    seed              = SEED,
    solver_model      = "gpt-5",
    critic_model      = "meta-llama/Llama-3.3-70B-Instruct-Turbo",
    embed_model       = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    labels            = ["C1","C2","C3","C4","C5"],
    frame_slots       = ["addressee","modality","action","scope","condition",
                         "exception","temporal","consequence"],
    match_strategies  = ["direct_textual","conceptual","distributed","absent"],
    retrieval_topk    = 12,
    sufficiency_rounds= 3,        # max query-expansion rounds
    sufficiency_delta = 0.15,     # stop when <15% of returned candidates are new
    max_critic_retries= 2,
    stability_runs    = 5,        # k for intra-run alpha (raise to 20 for the final run)
    stability_temp    = 0.7,
    negctrl_rate      = 0.15,
    concurrency       = 6,
    batch_flush       = 10,
    jurisdictions     = ["KSA","BHR","UAE","QAT","OMN","KWT","DIFC","ADGM"],
)
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True,
                                        ensure_ascii=False).encode()).hexdigest()[:16]
CONFIG["config_hash"] = CONFIG_HASH

MANIFEST = dict(run_tag=RUN_TAG, config=CONFIG,
                started=time.strftime("%Y-%m-%d %H:%M:%S"),
                python=sys.version.split()[0], device=DEVICE)
(DIRS["00_manifest"]/"manifest.json").write_text(
    json.dumps(MANIFEST, indent=2, ensure_ascii=False), encoding="utf-8")
print("config_hash:", CONFIG_HASH)

### 2.1 · Checkpoint store



In [ ]:
class Store:
    """Append-only JSONL keyed by record id. Re-running skips completed ids."""
    def __init__(self, path: Path):
        self.path = Path(path); self.path.parent.mkdir(parents=True, exist_ok=True)
        self._done, self._rows = set(), []
        if self.path.exists():
            for line in self.path.read_text(encoding="utf-8").splitlines():
                if not line.strip(): continue
                try:
                    r = json.loads(line); self._rows.append(r); self._done.add(r["id"])
                except Exception: pass
    def done(self, rid): return rid in self._done
    @property
    def n(self): return len(self._done)
    def add(self, rows):
        rows = [r for r in rows if r["id"] not in self._done]
        if not rows: return 0
        with self.path.open("a", encoding="utf-8") as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False)+"\n")
                self._done.add(r["id"]); self._rows.append(r)
        return len(rows)
    def df(self): return pd.DataFrame(self._rows)
    def rows(self): return list(self._rows)

def sha(*parts):
    return hashlib.sha256("||".join(map(str, parts)).encode("utf-8")).hexdigest()[:20]

TRACE = Store(DIRS["05_traces"]/"llm_calls.jsonl")
print("existing trace calls:", TRACE.n)

def purge_failures(store_path):
    p = Path(store_path)
    if not p.exists(): return None
    rows = [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]
    good = [r for r in rows if r.get("label")]
    if len(good) < len(rows):
        p.with_suffix(".jsonl.bak").write_text(
            "\n".join(json.dumps(r, ensure_ascii=False) for r in rows), encoding="utf-8")
        p.write_text("".join(json.dumps(r, ensure_ascii=False)+"\n" for r in good),
                     encoding="utf-8")
        print(f"{p.name}: kept {len(good)}, cleared {len(rows)-len(good)} failures for retry")
    return Store(p)


## 3 · Leakage control

Three artefact classes must not reach an agent prompt: the reference comparison documents, mutation ground truth, and negative-control identities. The guard fingerprints n-grams unique to the reference documents' comparative findings, after subtracting every n-gram occurring in a primary source, and screens each outgoing prompt. A match raises rather than warns.

In [ ]:
class LeakageError(RuntimeError): pass

# Sentences carrying a comparative FINDING are what must never reach an agent.
# The reference PDFs quote GDPR and statute text verbatim; quotation is NOT leakage.
VERDICT_MARKERS = [
    "does not explicitly", "does not expressly", "does not provide", "does not refer",
    "not provided for", "no equivalent", "unlike the gdpr", "in contrast to the gdpr",
    "is silent", "fairly consistent", "fairly inconsistent", "inconsistent", "consistent:",
    "less detailed", "more restrictive", "narrower than", "broader than", "differs from",
    "whereas the gdpr", "while the gdpr", "the pdpl does", "the law does not",
    "لا ينص", "لا يشير", "لا يتضمن", "لا يوجد ما يقابل", "بخلاف اللائحة", "على عكس",
]

class LeakageGuard:
    """Blocks the reference documents' ANALYTICAL FINDINGS from any agent prompt.

    Only sentences expressing a comparative verdict are fingerprinted, at 12-gram
    resolution, after subtracting every n-gram that also occurs in a primary source
    (GDPR or a target statute). Legal boilerplate therefore cannot trip the guard."""

    def __init__(self, ngram=12, min_len=40, hit_threshold=2):
        self.ngram, self.min_len, self.hit_threshold = ngram, min_len, hit_threshold
        self.fingerprints, self.primary, self.secrets = set(), set(), set()
        self.enabled, self.checks, self.n_verdict_sents = True, 0, 0

    @staticmethod
    def _norm(t):
        t = re.sub(r"[\u064B-\u0652\u0640]", "", str(t))
        t = re.sub(r"[^\w\u0621-\u064A]+", " ", t)
        return re.sub(r"\s+", " ", t).strip().lower()

    def _grams(self, text):
        w = self._norm(text).split()
        return {hashlib.blake2b(" ".join(w[i:i+self.ngram]).encode(), digest_size=8).hexdigest()
                for i in range(0, max(0, len(w)-self.ngram+1))}

    def register_primary(self, text, name=""):
        self.primary |= self._grams(text)

    def register_reference(self, text, name=""):
        sents = re.split(r"(?<=[.!?\u061F\u06D4])\s+|\n{2,}", text)
        verdict = [s for s in sents
                   if len(s.split()) >= self.ngram
                   and any(m in s.lower() for m in VERDICT_MARKERS)]
        g = set()
        for s in verdict: g |= self._grams(s)
        uniq = g - self.primary
        self.fingerprints |= uniq
        self.n_verdict_sents += len(verdict)
        print(f"  guard: {name} -> {len(sents):,} sentences, {len(verdict):,} carry a verdict, "
              f"{len(uniq):,} fingerprints")

    def finalize(self):
        before = len(self.fingerprints)
        self.fingerprints -= self.primary
        print(f"  guard: finalize -> {len(self.fingerprints):,} active fingerprints from "
              f"{self.n_verdict_sents:,} verdict sentences "
              f"(removed {before-len(self.fingerprints):,} overlapping)")

    def register_secret(self, s):
        if s and len(str(s)) >= 3: self.secrets.add(str(s))

    def check(self, prompt, where=""):
        if not self.enabled: return prompt
        self.checks += 1
        for s in self.secrets:
            if s in prompt:
                raise LeakageError(f"[{where}] ground-truth secret leaked into prompt: {s!r}")
        if len(prompt) >= self.min_len and self.fingerprints:
            hits = self._grams(prompt) & self.fingerprints
            if len(hits) >= self.hit_threshold:
                raise LeakageError(
                    f"[{where}] prompt reproduces a reference comparative finding "
                    f"({len(hits)} {self.ngram}-gram hits >= {self.hit_threshold}). "
                    f"Reference verdicts must not enter agent context.")
        return prompt

GUARD = LeakageGuard()
print(f"LeakageGuard ready — verdict-sentence fingerprinting at {GUARD.ngram}-gram resolution")

## 4 · Model clients

Batched, retried, and traced. Every call is persisted with its prompt hash so any reported figure can be recomputed from the trace.

In [ ]:
from openai import OpenAI, AsyncOpenAI
from together import AsyncTogether

def _secret(name):
    v = os.environ.get(name)
    if v: return v
    try: return userdata.get(name)
    except Exception: return None

OPENAI_KEY   = _secret("OPENAI_API_KEY")
TOGETHER_KEY = _secret("TOGETHER_API_KEY") or _secret("TOGETHERAI_API_KEY")
assert OPENAI_KEY,   "OPENAI_API_KEY not found in env or Colab secrets"
assert TOGETHER_KEY, "TOGETHER_API_KEY not found in env or Colab secrets"

aoai      = AsyncOpenAI(api_key=OPENAI_KEY)
atogether = AsyncTogether(api_key=TOGETHER_KEY)
SEM = asyncio.Semaphore(CONFIG["concurrency"])

async def _retry(fn, tries=5, base=2.0):
    for a in range(tries):
        try: return await fn()
        except Exception as e:
            if a == tries-1: raise
            await asyncio.sleep(base*(2**a) + random.random())

async def call_solver(system, user, *, temperature=None, seed=None, tag=""):
    GUARD.check(system, "solver.system"); GUARD.check(user, "solver.user")
    key = sha(CONFIG["solver_model"], system, user, temperature, seed, tag)
    kw = dict(model=CONFIG["solver_model"],
              messages=[{"role":"system","content":system},{"role":"user","content":user}],
              response_format={"type":"json_object"})
    if temperature is not None: kw["temperature"] = temperature
    if seed is not None:        kw["seed"] = seed
    async with SEM:
        async def go(): return await aoai.chat.completions.create(**kw)
        try:
            r = await _retry(go)
        except Exception as e:                    # gpt-5 may reject temperature/seed
            kw.pop("temperature", None); kw.pop("seed", None)
            r = await _retry(lambda: aoai.chat.completions.create(**kw))
    txt = r.choices[0].message.content
    TRACE.add([dict(id=key, role="solver", model=CONFIG["solver_model"], tag=tag,
                    temperature=temperature, seed=seed,
                    prompt_hash=sha(system,user), output=txt,
                    usage=getattr(r,"usage",None).model_dump() if getattr(r,"usage",None) else {},
                    ts=time.time())])
    return txt, key

async def call_critic(system, user, *, temperature=0.0, tag=""):
    GUARD.check(system, "critic.system"); GUARD.check(user, "critic.user")
    key = sha(CONFIG["critic_model"], system, user, temperature, tag)
    async with SEM:
        async def go():
            return await atogether.chat.completions.create(
                model=CONFIG["critic_model"], temperature=temperature, max_tokens=1600,
                messages=[{"role":"system","content":system},{"role":"user","content":user}])
        r = await _retry(go)
    txt = r.choices[0].message.content
    TRACE.add([dict(id=key, role="critic", model=CONFIG["critic_model"], tag=tag,
                    temperature=temperature, prompt_hash=sha(system,user),
                    output=txt, ts=time.time())])
    return txt, key

def parse_json(txt, default=None):
    if txt is None: return default
    m = re.search(r"\{.*\}", txt, re.S)
    if not m: return default
    try: return json.loads(m.group(0))
    except Exception:
        try: return json.loads(re.sub(r",\s*([}\]])", r"\1", m.group(0)))
        except Exception: return default
print("clients ready")

## 5 · Corpus ingestion

GCC statutes are read from a curated  table; the GDPR is parsed from its PDF text layer. The dual-source Arabic merge remains available for PDF sources: the embedded text layer reverses the lam-alef ligature and reorders parenthesised cross-references under bidirectional layout, which would corrupt modal operators and propagate into deontic classification.

In [ ]:
import fitz, subprocess, difflib, shutil
try:
    import pytesseract
    from PIL import Image
except Exception:
    pytesseract = None; Image = None

AR = "\u0621-\u064A\u0640\u064B-\u0652"
_clean = lambda s: re.sub(r"[\u202a-\u202e\u200e\u200f\u2066-\u2069]", "", s or "")

# ---- capability probe: never assume a binary is present ----
HAVE_PDFTOTEXT = shutil.which("pdftotext") is not None
HAVE_TESS      = shutil.which("tesseract") is not None and pytesseract is not None
HAVE_ARA = False
if HAVE_TESS:
    try:
        _r = subprocess.run(["tesseract","--list-langs"], capture_output=True)
        HAVE_ARA = "ara" in (_r.stdout + _r.stderr).decode().split()
    except Exception:
        HAVE_ARA = False
print(f"extractor capabilities: pdftotext={HAVE_PDFTOTEXT} tesseract={HAVE_TESS} arabic={HAVE_ARA}")
if not HAVE_PDFTOTEXT:
    print("  note: falling back to PyMuPDF block extraction for the text layer")

def _norm_lig(w):
    w = re.sub(r"[\u064B-\u0652\u0640]", "", w)
    for a in "\u0622\u0623\u0625": w = w.replace(a, "\u0627")
    w = w.replace("\u0629","\u0647").replace("\u0649","\u064a")
    return w.replace("\u0644\u0627","@").replace("\u0627\u0644","@")

_TOK = re.compile(f"[{AR}]+|[0-9][0-9,]*|[^\\s{AR}0-9]")
def _seq(text):
    items, follow, pend = [], [], ""
    for m in _TOK.finditer(text):
        v = m.group(0)
        if re.fullmatch(f"[{AR}]+|[0-9][0-9,]*", v):
            if items: follow[-1] = pend
            items.append(v); follow.append(""); pend = ""
        else: pend += v
    if follow: follow[-1] = pend
    return items, follow

_CFIX = {"\u00bb":"\u060c","\u00ab":"\u060c","\u061b":"\u060c"}
_pfix = lambda s: "".join(_CFIX.get(c,c) for c in s)

def _cands(w):
    out = {w}
    out.add(re.sub("([\u0627\u0623\u0625\u0622])\u0627\u0644","\\1\u0644\u0627", w))
    out.add(re.sub("\u0644([\u0627\u0623\u0625\u0622])\u0644","\u0644\u0644\\1", w))
    if re.fullmatch("[\u0648\u0641]?\u0627\u0644", w): out.add(w[:-2]+"\u0644\u0627")
    return out

def _layer_text(pdf: Path, page: int, doc):
    """Text layer for one page. pdftotext when available (best punctuation); else PyMuPDF blocks."""
    if HAVE_PDFTOTEXT:
        try:
            return _clean(subprocess.run(
                ["pdftotext","-f",str(page+1),"-l",str(page+1),str(pdf),"-"],
                capture_output=True).stdout.decode("utf-8","ignore"))
        except Exception:
            pass
    blocks = [b for b in doc[page].get_text("blocks") if b[6] == 0 and b[4].strip()]
    blocks.sort(key=lambda b: (round(b[1], 1), -b[2]))      # top-down, right-to-left
    return _clean("\n".join(b[4].strip() for b in blocks))

def extract_pdf(pdf: Path, ocr_lang="ara", dpi=300, use_ocr=True):
    """Returns (merged_text, per-page stats). Degrades gracefully when a binary is absent."""
    use_ocr = use_ocr and HAVE_ARA
    doc = fitz.open(str(pdf)); pages, stats = [], []
    for p in range(doc.page_count):
        layer = _layer_text(pdf, p, doc)
        ocr = ""
        if use_ocr:
            try:
                pix = doc[p].get_pixmap(dpi=dpi)
                img = Image.open(io.BytesIO(pix.tobytes("png")))
                ocr = _clean(pytesseract.image_to_string(img, lang=ocr_lang, config="--psm 6"))
            except Exception:
                ocr = ""
        if not ocr.strip():
            pages.append(layer)
            stats.append(dict(page=p+1, matched=0, fallback=len(layer.split()),
                              mode="layer_only" if HAVE_PDFTOTEXT else "pymupdf_only"))
            continue
        li, lf = _seq(layer); oi, of = _seq(ocr)
        lex = set()
        for w in re.findall(f"[{AR}]+", ocr):
            lex.add(w)
            if w.endswith("\u0621") and len(w) > 3: lex.add(w[:-1])
        sm = difflib.SequenceMatcher(a=[_norm_lig(x) for x in oi],
                                     b=[_norm_lig(x) for x in li], autojunk=False)
        parts, eq, ne = [], 0, 0
        for tag,i1,i2,j1,j2 in sm.get_opcodes():
            if tag == "equal":
                for d in range(i2-i1):
                    parts.append(oi[i1+d] + _pfix(lf[j1+d])); eq += 1
            else:
                for j in range(j1, j2):
                    w = li[j]
                    for c in _cands(w):
                        if c != w and c in lex: w = c; break
                    parts.append(w + _pfix(lf[j])); ne += 1
        stats.append(dict(page=p+1, matched=eq, fallback=ne, mode="merged"))
        pages.append(" ".join(parts))
    doc.close()
    txt = "\n".join(pages)
    txt = re.sub(r"\s+([\u060c\u061f.:,)\]])", r"\1", txt)
    txt = re.sub(r"([(\[])\s+", r"\1", txt)
    txt = re.sub(r"https?://\S+", " ", txt)
    txt = re.sub(r"[ \t]{2,}", " ", txt)
    return txt, stats

# ---- article splitting: ASCII + Arabic-Indic digits + word ordinals, no strict-sequence filter ----
_IND2ASCII = str.maketrans("\u0660\u0661\u0662\u0663\u0664\u0665\u0666\u0667\u0668\u0669",
                           "0123456789")
ORDINALS = ["الأولى","الثانية","الثالثة","الرابعة","الخامسة","السادسة","السابعة","الثامنة",
            "التاسعة","العاشرة","الحادية عشرة","الثانية عشرة","الثالثة عشرة","الرابعة عشرة",
            "الخامسة عشرة","السادسة عشرة","السابعة عشرة","الثامنة عشرة","التاسعة عشرة","العشرون"]
ORD2NUM = {o: i+1 for i, o in enumerate(ORDINALS)}

AR_ART = re.compile(r"(?:^|[\s\n])(?:ال)?مادة\s*[\(\)]*\s*(?:رقم\s*)?"
                    r"(?:([0-9\u0660-\u0669]{1,3})|(" + "|".join(ORDINALS) + r"))\s*[\)\:\-]?")
EN_ART = re.compile(r"(?:^|\n)\s*Article\s+(\d{1,3})\b")

def split_articles(text, lang="ar", strict_sequence=False):
    if lang == "en":
        marks = [(m.start(), m.end(), int(m.group(1))) for m in EN_ART.finditer(text)]
    else:
        marks = []
        for m in AR_ART.finditer(text):
            n = int(m.group(1).translate(_IND2ASCII)) if m.group(1) else ORD2NUM.get(m.group(2))
            if n: marks.append((m.start(), m.end(), n))
    if strict_sequence:
        keep, expect = [], 1
        for s, e, n in marks:
            if n == expect: keep.append((s, e, n)); expect += 1
    else:
        best = {}
        for s, e, n in marks: best[n] = (s, e, n)      # last occurrence wins
        keep = sorted(best.values(), key=lambda t: t[0])
    out = []
    for i, (s, e, n) in enumerate(keep):
        stop = keep[i+1][0] if i+1 < len(keep) else len(text)
        seg = re.sub(r"\s+", " ", text[e:stop]).strip(" .،:-")
        if len(seg) > 25: out.append(dict(article=n, text=seg))
    return out

print("ingestion utilities ready")

In [ ]:
# ---- discover inputs in EXP-1 ----
ANNT_CANDIDATES = [p for p in DATA_ROOT.iterdir()
                   if p.suffix.lower() in (".csv",".xlsx",".xls")
                   and "annt" in p.stem.lower().replace("_","").replace("-","").replace(" ","")]
if not ANNT_CANDIDATES:
    ANNT_CANDIDATES = [p for p in DATA_ROOT.iterdir() if p.suffix.lower() in (".csv",".xlsx",".xls")]
assert ANNT_CANDIDATES, f"No CSV/Excel found in {DATA_ROOT}"
ANNT_PATH = ANNT_CANDIDATES[0]
print("annotation table →", ANNT_PATH.name)

PDFS = sorted(DATA_ROOT.glob("*.pdf"))
print(f"\n{len(PDFS)} PDFs:")
for p in PDFS: print("   ", p.name)

def _guess(name):
    n = name.lower()
    if "comparison" in n or ("gdpr" in n and ("gcc" in n or " vs" in n or "v_" in n or "middle" in n)):
        return "REFERENCE"
    if "gdpr" in n or "679" in n: return "GDPR"
    return "OTHER"

FILEMAP = pd.DataFrame([dict(file=p.name, path=str(p), role=_guess(p.name)) for p in PDFS])
FILEMAP["kind"] = FILEMAP.role.map({"GDPR":"gdpr","REFERENCE":"reference_kb"}).fillna("ignore")
FILEMAP.to_csv(DIRS["01_corpus"]/"file_map.csv", index=False)
display(FILEMAP)

n_gdpr = (FILEMAP.kind=="gdpr").sum(); n_ref = (FILEMAP.kind=="reference_kb").sum()
print(f"\ngdpr={n_gdpr}  reference_kb={n_ref}  ignored={(FILEMAP.kind=='ignore').sum()}")
print(">>> Fix any mis-assignment BEFORE continuing (a reference PDF read as 'ignore' means the")
print("    leakage guard will not fingerprint it):")
print("    FILEMAP.loc[FILEMAP.file=='<name>', ['role','kind']] = ['REFERENCE','reference_kb']")
assert n_gdpr >= 1, "No GDPR PDF identified"
assert n_ref  >= 1, "No reference/comparison PDF identified — leakage guard would be empty"

In [ ]:
# ---- ingest GCC statutes from the annotation table + GDPR from PDF (checkpointed) ----
CORPUS  = Store(DIRS["01_corpus"]/"corpus.jsonl")
QUALITY = Store(DIRS["01_corpus"]/"extraction_quality.jsonl")

# ---------- 1. the curated statute table ----------
if ANNT_PATH.suffix.lower() == ".csv":
    ann = pd.read_csv(ANNT_PATH, encoding="utf-8-sig")
else:
    ann = pd.read_excel(ANNT_PATH)
print("raw table:", ann.shape); print("columns:", list(ann.columns))

def _canon(c): return re.sub(r"[^a-z]", "", str(c).lower())
COLMAP = {}
for c in ann.columns:
    k = _canon(c)
    if k in ("articlenumber","article","articleno","art"):        COLMAP[c] = "article_number"
    elif k in ("country","jurisdiction","state"):                 COLMAP[c] = "Country"
    elif k in ("sourcedocument","source","document","law","instrument"): COLMAP[c] = "Source_Document"
    elif k in ("content","text","articletext","body"):            COLMAP[c] = "content"
ann = ann.rename(columns=COLMAP)
missing = {"article_number","Country","Source_Document","content"} - set(ann.columns)
assert not missing, f"Missing columns after mapping: {missing}\nSeen: {list(ann.columns)}"

# normalise country -> ISO-ish codes used throughout the notebook
CTRY = {"saudi":"KSA","ksa":"KSA","saudiarabia":"KSA","السعودية":"KSA","المملكة":"KSA",
        "bahrain":"BHR","bahrin":"BHR","البحرين":"BHR",
        "uae":"UAE","unitedarabemirates":"UAE","emirates":"UAE","الإمارات":"UAE","الامارات":"UAE",
        "qatar":"QAT","قطر":"QAT", "oman":"OMN","عمان":"OMN","سلطنةعمان":"OMN",
        "kuwait":"KWT","الكويت":"KWT", "difc":"DIFC","adgm":"ADGM"}
def _code(x):
    k = re.sub(r"[^a-z\u0621-\u064A]", "", str(x).lower())
    return CTRY.get(k, str(x).strip().upper()[:4])
ann["jurisdiction"] = ann.Country.map(_code)

def _artnum(x):
    s = str(x).translate(str.maketrans("٠١٢٣٤٥٦٧٨٩","0123456789"))
    m = re.search(r"\d{1,3}", s)
    return int(m.group(0)) if m else None
ann["article"] = ann.article_number.map(_artnum)

ann["content"] = ann.content.astype(str).map(lambda t: re.sub(r"\s+", " ", t).strip())
before = len(ann)
ann = ann[ann.content.str.len() > 25].dropna(subset=["article"]).copy()
ann["article"] = ann.article.astype(int)
print(f"usable rows: {len(ann)} / {before}")

rows = []
for _, r in ann.iterrows():
    rid = sha(r.jurisdiction, r.article, r.Source_Document)
    rows.append(dict(id=rid, jurisdiction=r.jurisdiction, kind="statute",
                     source_file=str(r.Source_Document), article=int(r.article),
                     article_label=str(r.article_number), text=r.content,
                     n_chars=len(r.content)))
CORPUS.add(rows)

for (j, src), g in ann.groupby(["jurisdiction","Source_Document"]):
    fid = sha("tbl", j, src)
    if QUALITY.done(fid): continue
    QUALITY.add([dict(id=fid, file=str(src), role=j, pages=None, articles=len(g),
                      matched=int(g.content.str.split().str.len().sum()), fallback=0,
                      align_rate=1.0, mode="curated_table")])

# ---------- 2. GDPR from PDF ----------
for _, row in FILEMAP[FILEMAP.kind.eq("gdpr")].iterrows():
    fid = sha(row.file)
    if QUALITY.done(fid): continue
    txt, stats = extract_pdf(Path(row.path), use_ocr=False)
    arts = split_articles(txt, lang="en")
    CORPUS.add([dict(id=sha("GDPR", a["article"]), jurisdiction="GDPR", kind="gdpr",
                     source_file=row.file, article=a["article"],
                     article_label=f"Article {a['article']}", text=a["text"],
                     n_chars=len(a["text"])) for a in arts])
    QUALITY.add([dict(id=fid, file=row.file, role="GDPR", pages=len(stats), articles=len(arts),
                      matched=0, fallback=sum(s["fallback"] for s in stats),
                      align_rate=None, mode="layer_only")])

# ---------- 3. report ----------
corpus_df = CORPUS.df()
assert len(corpus_df) > 0, "Nothing ingested."
print(f"\ncorpus: {len(corpus_df)} articles | statutes={(corpus_df.kind=='statute').sum()} "
      f"| gdpr={(corpus_df.kind=='gdpr').sum()}")
display(QUALITY.df()[["file","role","articles","mode"]])
display(corpus_df.groupby(["kind","jurisdiction"])
        .agg(articles=("article","count"), mean_chars=("n_chars","mean")).round(0))

assert (corpus_df.kind=="gdpr").sum() >= 50, "GDPR article split looks wrong (<50 articles)"
_thin = corpus_df[corpus_df.kind.eq("statute")].groupby("jurisdiction").size()
if (_thin < 5).any():
    print("\n⚠ jurisdictions with <5 articles — check the table:",
          _thin[_thin < 5].to_dict())

## 6 · Reference knowledge base

Published comparative analyses are parsed here and registered with the leakage guard, then isolated from the inference path. The self-test verifies that primary statutory text passes and reference analytical prose is blocked.

In [ ]:
# ---- 1. PRIMARY sources first: GDPR + statutes are PERMITTED in agent prompts ----
assert "corpus_df" in dir() and len(corpus_df) > 0, "Run the ingestion cells first."
for _, r in tqdm(corpus_df.iterrows(), total=len(corpus_df), desc="primary n-grams"):
    GUARD.primary |= GUARD._grams(r.text)
print(f"primary n-grams (GDPR + statutes): {len(GUARD.primary):,}")

# ---- 2. reference KB: fingerprint ONLY its unique analytical content ----
REF_TEXT = {}
_ref_rows = FILEMAP[FILEMAP.kind.eq("reference_kb")]
assert len(_ref_rows) > 0, "No reference_kb PDFs — the guard would have nothing to protect."
for _, row in _ref_rows.iterrows():
    t, _ = extract_pdf(Path(row.path), use_ocr=False)
    REF_TEXT[row.file] = t
    GUARD.register_reference(t, row.file)
GUARD.finalize()

(DIRS["03_reference_kb"]/"reference_raw.json").write_text(
    json.dumps(REF_TEXT, ensure_ascii=False, indent=1), encoding="utf-8")

# ---- 3. self-test: primary text must PASS, reference analysis must BLOCK ----
_ok_pass = _ok_block = None
_gdpr = corpus_df[corpus_df.kind.eq("gdpr")]
if len(_gdpr):
    try:
        GUARD.check(_gdpr.iloc[0].text[:1500], "selftest.gdpr")
        _ok_pass = True;  print("PASS  GDPR text is not blocked (correct)")
    except LeakageError as e:
        _ok_pass = False; print("FAIL  GDPR text still blocked ->", str(e)[:140])

_ref = list(REF_TEXT.values())[0]
_probe = next((_ref[i:i+1200] for i in range(0, max(1, len(_ref)-1200), 1200)
               if len(GUARD._grams(_ref[i:i+1200]) & GUARD.fingerprints) >= GUARD.hit_threshold), None)
if _probe:
    try:
        GUARD.check(_probe, "selftest.ref")
        _ok_block = False; print("FAIL  reference prose NOT blocked (guard too weak)")
    except LeakageError:
        _ok_block = True;  print("PASS  reference analytical prose is blocked (correct)")
else:
    _ok_block = None
    print("WARN  no reference chunk carries >= %d unique n-grams." % GUARD.hit_threshold)
    print("      The reference may be almost entirely verbatim quotation. If so, say so in the")
    print("      methods rather than claiming leakage protection the corpus does not require.")

GUARD_SELFTEST = dict(primary_passes=_ok_pass, reference_blocked=_ok_block,
                      fingerprints=len(GUARD.fingerprints), primary=len(GUARD.primary),
                      hit_threshold=GUARD.hit_threshold)
(DIRS["03_reference_kb"]/"guard_selftest.json").write_text(
    json.dumps(GUARD_SELFTEST, indent=2), encoding="utf-8")
print(f"\nreference sources: {len(REF_TEXT)} | active fingerprints: {len(GUARD.fingerprints):,} "
      f"| primary: {len(GUARD.primary):,}")

## 7 · Requirement units — the measurement instrument

The GDPR is decomposed into atomic requirement units, each an eight-slot deontic frame. The instrument is generated once, verified, and frozen.

In [ ]:
RU_PATH  = DIRS["02_requirement_units"]/"requirement_units.json"
RU_FROZEN = RU_PATH.exists()

RU_SYS = (
 "You are a legal-informatics analyst decomposing the EU GDPR into atomic normative "
 "requirement units for comparative analysis. Output STRICT JSON only.")

RU_USER_TMPL = """Decompose the following GDPR article text into atomic requirement units.
A requirement unit is ONE normative proposition (one addressee + one deontic modality + one action).
Split compound obligations. Ignore purely declaratory or definitional text unless it constrains a norm.

For each unit return the eight-slot frame. Use null for a slot the text does not specify.

Return JSON: {{"units":[{{"ru_label":"<short name>","gdpr_article":"{art}",
"addressee":..., "modality":"obligation|prohibition|permission|right|power",
"action":..., "scope":..., "condition":..., "exception":..., "temporal":..., "consequence":...,
"proposition":"<one sentence stating the norm>"}}]}}

GDPR ARTICLE {art}:
<<<{text}>>>"""

async def build_requirement_units():
    gdpr = corpus_df[corpus_df.kind.eq("gdpr")].sort_values("article")
    assert len(gdpr) > 0, "No GDPR articles ingested — check file_map role assignment."
    store = Store(DIRS["02_requirement_units"]/"_ru_raw.jsonl")
    async def one(r):
        rid = sha("ru", r.article)
        if store.done(rid): return
        txt, _ = await call_solver(RU_SYS, RU_USER_TMPL.format(art=r.article, text=r.text[:9000]),
                                   temperature=0.0, seed=SEED, tag=f"ru:{r.article}")
        d = parse_json(txt, {"units":[]})
        store.add([dict(id=rid, article=int(r.article), units=d.get("units", []))])
    rows = list(gdpr.itertuples())
    for i in range(0, len(rows), 12):
        await asyncio.gather(*[one(r) for r in rows[i:i+12]])
        print(f"  RU {min(i+12,len(rows))}/{len(rows)}", end="\r")
    units = []
    for rec in sorted(store.rows(), key=lambda x: x["article"]):
        for u in rec["units"]:
            u = dict(u); u["gdpr_article"] = rec["article"]
            u["ru_id"] = f"RU-{len(units)+1:03d}"
            units.append(u)
    return units

if not RU_FROZEN:
    UNITS = asyncio.get_event_loop().run_until_complete(build_requirement_units())
    RU_PATH.write_text(json.dumps(dict(config_hash=CONFIG_HASH, version=1, units=UNITS),
                                  ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"\nGenerated {len(UNITS)} requirement units → REVIEW THEN RE-RUN THIS CELL to freeze.")
else:
    UNITS = json.loads(RU_PATH.read_text(encoding="utf-8"))["units"]
    print(f"Instrument FROZEN: {len(UNITS)} requirement units (v1, {CONFIG_HASH})")

RU_DF = pd.DataFrame(UNITS)
display(RU_DF[["ru_id","gdpr_article","ru_label","modality"]].head(12))
print(RU_DF.modality.value_counts().to_dict())

## 8 · Retrieval index

Multilingual sentence embeddings over the statute corpus, indexed for inner-product search and filtered by jurisdiction at query time.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

EMB = SentenceTransformer(CONFIG["embed_model"], device=DEVICE)
EMB.max_seq_length = 256

IDX_PATH = DIRS["04_index"]/"index.faiss"
META_PATH= DIRS["04_index"]/"index_meta.parquet"

TARGETS = corpus_df[corpus_df.kind.eq("statute")].reset_index(drop=True)

if IDX_PATH.exists() and META_PATH.exists():
    index = faiss.read_index(str(IDX_PATH)); IDXMETA = pd.read_parquet(META_PATH)
    print("index loaded:", index.ntotal)
else:
    vecs = EMB.encode(TARGETS.text.tolist(), batch_size=64, convert_to_numpy=True,
                      normalize_embeddings=True, show_progress_bar=True)
    index = faiss.IndexFlatIP(vecs.shape[1]); index.add(vecs.astype("float32"))
    faiss.write_index(index, str(IDX_PATH))
    IDXMETA = TARGETS[["id","jurisdiction","article","text"]].copy()
    IDXMETA.to_parquet(META_PATH)
    np.save(DIRS["04_index"]/"vecs.npy", vecs)
    print("index built:", index.ntotal)

VECS = np.load(DIRS["04_index"]/"vecs.npy") if (DIRS["04_index"]/"vecs.npy").exists() else None

def retrieve(query, jurisdiction, k=None, exclude_ids=()):
    k = k or CONFIG["retrieval_topk"]
    q = EMB.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    D,I = index.search(q, min(len(IDXMETA), k*8))
    out = []
    for score, i in zip(D[0], I[0]):
        r = IDXMETA.iloc[i]
        if r.jurisdiction != jurisdiction: continue
        if r.id in exclude_ids: continue
        out.append(dict(id=r.id, article=int(r.article), text=r.text, score=float(score)))
        if len(out) >= k: break
    return out
print("retriever ready")

## 9 · Agents

Four roles execute under deterministic orchestration. The Solver emits candidate selection, exclusions with reasons, the target frame with spans and citations, slot adjudication, and only then a holistic judgement — the verdict last, so that slot analysis cannot become post-hoc rationalisation. The Critic receives the record without the Solver's reasoning and attempts falsification, on a different model family.

In [ ]:
SOLVER_SYS = """You are a comparative-law analyst performing PROVISION-LEVEL TEXTUAL GAP ANALYSIS.

You compare ONE GDPR requirement unit against candidate provisions from ONE target jurisdiction's
statute. You assess ONLY what the texts say. You never assess enforcement, regulatory practice,
case law, or real-world compliance.

Functional equivalence governs: a counterpart need NOT use GDPR terminology or structure. A norm may
be assembled from a definition plus several operative articles. Lexical difference alone is never a gap.

Work in this order and emit the JSON keys in this order:
 1. candidates_selected  - ids of provisions that bear on the requirement
 2. candidates_excluded  - id + reason for every candidate you rejected
 3. target_frame         - the eight slots as expressed by the TARGET text, each with a verbatim
                           span and the article it came from (null span if the slot is unexpressed)
 4. slot_verdicts        - per slot: gdpr value, target value, basis, verdict, citation
 5. holistic_label       - your own overall judgement (used only as a consistency signal)
 6. confidence           - 0.0-1.0
 7. reasoning_note       - <= 3 sentences

slot verdict vocabulary: covered | weaker_modality | narrower | broader | partial | uncovered
basis vocabulary:        direct_textual | conceptual | distributed | absent

Output STRICT JSON only."""

SOLVER_USER = """GDPR REQUIREMENT UNIT [{ru_id}] (GDPR Art. {art})
proposition : {proposition}
addressee   : {addressee}
modality    : {modality}
action      : {action}
scope       : {scope}
condition   : {condition}
exception   : {exception}
temporal    : {temporal}
consequence : {consequence}

TARGET JURISDICTION: {juris}
CANDIDATE PROVISIONS:
{candidates}

Return the JSON object."""

def fmt_candidates(cands):
    return "\n\n".join(f"[{c['id']}] Article {c['article']}\n{c['text'][:2200]}" for c in cands) \
           or "(no candidate provisions retrieved)"

async def solver_step(ru, juris, cands, *, temperature=None, seed=None, tag=""):
    user = SOLVER_USER.format(
        ru_id=ru["ru_id"], art=ru.get("gdpr_article"),
        proposition=ru.get("proposition"), addressee=ru.get("addressee"),
        modality=ru.get("modality"), action=ru.get("action"), scope=ru.get("scope"),
        condition=ru.get("condition"), exception=ru.get("exception"),
        temporal=ru.get("temporal"), consequence=ru.get("consequence"),
        juris=juris, candidates=fmt_candidates(cands))
    txt, key = await call_solver(SOLVER_SYS, user, temperature=temperature, seed=seed, tag=tag)
    return parse_json(txt, {}) or {}, key

In [ ]:
CRITIC_SYS = """You are an adversarial reviewer of comparative-law gap analyses. You did NOT see the
analyst's reasoning. Your task is to FALSIFY the analyst's verdict using only the provisions supplied.

Scrutinise most severely any claim that NO counterpart exists (label C5): check whether a counterpart
is present under different terminology, or distributed across several provisions or a definition.

Also check: (a) does every cited span actually appear in the quoted provision, (b) is the deontic
modality claim supported by the modal wording, (c) is any slot verdict unsupported by its citation.

Return STRICT JSON:
{"verdict":"uphold"|"revise","objection":"<specific, actionable defect or empty>",
 "counter_evidence":[{"id":"<provision id>","span":"<verbatim>","why":"..."}],
 "confidence":0.0-1.0}"""

CRITIC_USER = """GDPR REQUIREMENT UNIT [{ru_id}]
proposition: {proposition}
modality   : {modality}

TARGET JURISDICTION: {juris}
ANALYST LABEL: {label}
ANALYST SLOT VERDICTS: {slots}
ANALYST CITATIONS: {cites}

ALL PROVISIONS AVAILABLE (the analyst saw exactly these):
{candidates}

Falsify or uphold. Return the JSON object."""

async def critic_step(ru, juris, cands, record, tag=""):
    user = CRITIC_USER.format(
        ru_id=ru["ru_id"], proposition=ru.get("proposition"), modality=ru.get("modality"),
        juris=juris, label=record.get("label"),
        slots=json.dumps(record.get("slot_verdicts", []), ensure_ascii=False)[:2500],
        cites=json.dumps([s.get("citation") for s in record.get("slot_verdicts", [])],
                         ensure_ascii=False)[:900],
        candidates=fmt_candidates(cands))
    txt, key = await call_critic(CRITIC_SYS, user, tag=tag)
    return parse_json(txt, {"verdict":"uphold","objection":"","counter_evidence":[]}) or {}, key

### 9.1 · Deterministic verdict rule

The aggregate label is computed by rule over the slot verdicts rather than generated, so identical slot analyses always yield an identical label. `normalize_slots` absorbs the shape variation the model returns.

In [ ]:
SLOT_W = {"addressee":1.0,"modality":1.0,"action":1.0,"scope":1.0,
          "condition":0.7,"exception":0.7,"temporal":0.7,"consequence":0.7}

def normalize_slots(sv):
    """Coerce any shape the model returns into list[dict(slot, verdict, ...)]."""
    out = []
    if sv is None: return out
    if isinstance(sv, dict):
        for k, val in sv.items():
            if isinstance(val, dict):
                d = dict(val); d.setdefault("slot", k); out.append(d)
            else:
                out.append({"slot": k, "verdict": str(val)})
        return out
    if isinstance(sv, list):
        for s in sv:
            if isinstance(s, dict):
                out.append(s)
            elif isinstance(s, str) and ":" in s:          # "modality: weaker_modality"
                k, _, v = s.partition(":")
                out.append({"slot": k.strip(), "verdict": v.strip()})
        return out
    return out

def _as_ids(x):
    if x is None: return []
    if isinstance(x, str): return [x]
    if isinstance(x, dict): return [v.get("id", v) if isinstance(v, dict) else v
                                    for v in x.values()]
    return [i.get("id") if isinstance(i, dict) else i for i in x]

def derive_label(slot_verdicts, candidates_selected):
    """Aggregate gap label computed by rule over slot verdicts. Never generated."""
    sv = normalize_slots(slot_verdicts)
    v = {str(s.get("slot","")).lower(): str(s.get("verdict","")).lower() for s in sv}
    vals = [x for x in v.values() if x]
    if not vals or not _as_ids(candidates_selected): return "C5"
    if all(x == "uncovered" for x in vals):  return "C5"
    if v.get("modality") == "weaker_modality": return "C3"
    if v.get("addressee") in ("narrower","broader") or v.get("scope") in ("narrower","broader"):
        return "C4"
    if any(x in ("uncovered","partial") for x in vals): return "C2"
    return "C1"

def direction_of(slot_verdicts):
    v = {str(s.get("slot","")).lower(): str(s.get("verdict","")).lower()
         for s in normalize_slots(slot_verdicts)}
    for k in ("scope","addressee"):
        if v.get(k) in ("narrower","broader"): return v[k]
    return None

### 9.2 · Orchestrator

Routing is a state machine in code. Four metacognitive functions operate across it: retrieval sufficiency, strategy selection, slot completeness, and reflection on rule-versus-holistic agreement.

In [ ]:
EXPANSION_TMPL = ("Generate {n} short alternative Arabic/English search phrases that a lawyer would "
                  "use to find a provision expressing this norm, using DIFFERENT terminology. "
                  'Return JSON {{"queries":["...", ...]}}.\n\nNORM: {prop}')

async def expand_queries(prop, n=4, tag=""):
    txt, _ = await call_solver("You generate legal search queries. STRICT JSON only.",
                               EXPANSION_TMPL.format(n=n, prop=prop),
                               temperature=0.0, seed=SEED, tag=tag)
    return (parse_json(txt, {}) or {}).get("queries", [])[:n]

async def orchestrate(ru, juris, *, ablation=None, temperature=None, seed=None,
                      corpus_filter=None, tag=""):
    """Assess one requirement unit against one jurisdiction.

    ablation selects the configuration: None (full), 'no_sufficiency', 'no_strategy',
    'no_critic', 'single_agent'.
    """
    abl = ablation or "full"
    t0 = time.time(); n_calls = 0

    # STATE 1 — retrieval under the sufficiency criterion
    seen, cands, curve = set(), [], []
    q0 = ru.get("proposition") or ru.get("ru_label") or ""
    rounds = 1 if abl in ("no_sufficiency", "single_agent") else CONFIG["sufficiency_rounds"]
    queries = [q0]
    for rnd in range(rounds):
        new = 0
        for q in queries:
            for c in retrieve(q, juris, exclude_ids=corpus_filter or ()):
                if c["id"] not in seen:
                    seen.add(c["id"]); cands.append(c); new += 1
        curve.append(dict(round=rnd+1, n_queries=len(queries), new=new, total=len(cands)))
        if rnd+1 >= rounds: break
        if new/max(1, len(cands)) < CONFIG["sufficiency_delta"]: break   # saturated
        queries = await expand_queries(q0, tag=f"{tag}:exp{rnd}"); n_calls += 1
        if not queries: break
    cands = sorted(cands, key=lambda c: -c["score"])[:CONFIG["retrieval_topk"]]
    sufficiency_met = (len(curve) > 1 and
                       curve[-1]["new"]/max(1, curve[-1]["total"]) < CONFIG["sufficiency_delta"]
                       ) or abl in ("no_sufficiency", "single_agent")

    # STATE 2 — slot adjudication
    rec, _ = await solver_step(ru, juris, cands, temperature=temperature, seed=seed,
                               tag=f"{tag}:solve"); n_calls += 1
    rec["slot_verdicts"] = normalize_slots(rec.get("slot_verdicts"))
    rec["label"] = derive_label(rec["slot_verdicts"], rec.get("candidates_selected"))

    # STATE 3 — falsification under bounded retry
    objections, revisions = [], 0
    if abl not in ("no_critic", "single_agent"):
        for attempt in range(CONFIG["max_critic_retries"]):
            crit, _ = await critic_step(ru, juris, cands, rec, tag=f"{tag}:crit{attempt}")
            n_calls += 1
            if crit.get("verdict") != "revise" or not crit.get("objection"): break
            objections.append(crit)
            ce = json.dumps(crit.get("counter_evidence", []), ensure_ascii=False)[:1200]
            u = SOLVER_USER.format(
                ru_id=ru["ru_id"], art=ru.get("gdpr_article"), proposition=ru.get("proposition"),
                addressee=ru.get("addressee"), modality=ru.get("modality"), action=ru.get("action"),
                scope=ru.get("scope"), condition=ru.get("condition"), exception=ru.get("exception"),
                temporal=ru.get("temporal"), consequence=ru.get("consequence"),
                juris=juris, candidates=fmt_candidates(cands)) + (
                f"\n\nREVIEWER OBJECTION (address it specifically; do not merely repeat the prior "
                f"answer):\n{crit['objection']}\nCOUNTER-EVIDENCE: {ce}")
            txt, _ = await call_solver(SOLVER_SYS, u, temperature=temperature, seed=seed,
                                       tag=f"{tag}:revise{attempt}"); n_calls += 1
            rec2 = parse_json(txt, {}) or {}
            if rec2:
                rec2["slot_verdicts"] = normalize_slots(rec2.get("slot_verdicts"))
                rec2["label"] = derive_label(rec2["slot_verdicts"], rec2.get("candidates_selected"))
                rec = rec2; revisions += 1

    # STATE 4 — reflection and telemetry
    slots_done = {str(s.get("slot", "")).lower() for s in rec["slot_verdicts"]}
    rec.update(
        ru_id=ru["ru_id"], gdpr_article=ru.get("gdpr_article"), jurisdiction=juris,
        ablation=abl, direction=direction_of(rec["slot_verdicts"]),
        sufficiency_curve=curve, sufficiency_met=bool(sufficiency_met),
        slot_completeness=round(len(slots_done & set(CONFIG["frame_slots"]))/8, 3),
        rule_vs_holistic_agree=(str(rec.get("holistic_label", "")).upper() == rec["label"]),
        n_revisions=revisions, objections=objections,
        n_candidates=len(cands), candidate_ids=[c["id"] for c in cands],
        n_llm_calls=n_calls, latency_s=round(time.time()-t0, 2),
        temperature=temperature, seed=seed)
    return rec

print("orchestrator ready")

## 10 · Batched runner

Aborts above a 5% failure rate rather than continuing to spend budget on a systematic error.

In [ ]:
async def run_matrix(pairs, store: Store, *, ablation=None, temperature=None, seed=None,
                     tag="main", corpus_filter=None, desc="run"):
    """pairs: list of (ru_dict, jurisdiction, record_id, extra_meta)"""
    todo = [p for p in pairs if not store.done(p[2])]
    print(f"{desc}: {len(todo)} to do / {len(pairs)} total  (resuming from {store.n})")
    buf = []
    pbar = tqdm(total=len(todo), desc=desc)
    B = CONFIG["concurrency"]*2
    for i in range(0, len(todo), B):
        chunk = todo[i:i+B]
        async def one(ru, juris, rid, extra):
            try:
                rec = await orchestrate(ru, juris, ablation=ablation, temperature=temperature,
                                        seed=seed, corpus_filter=corpus_filter,
                                        tag=f"{tag}:{rid[:8]}")
            except LeakageError: raise
            except Exception as e:
                rec = dict(ru_id=ru["ru_id"], jurisdiction=juris, label=None, error=str(e)[:300])
            rec["id"] = rid; rec.update(extra or {}); return rec
        res = await asyncio.gather(*[one(*c) for c in chunk], return_exceptions=False)
        buf.extend(res); pbar.update(len(chunk))
        if len(buf) >= CONFIG["batch_flush"]:
            store.add(buf); buf = []

    if buf: store.add(buf)
    pbar.close()
    _d = store.df()
    if len(_d) and "label" in _d.columns:
        fail = float(_d.label.isna().mean())
        print(f"{desc}: failure rate {fail:.1%}")
        if fail > 0.05:
            errs = _d[_d.label.isna()].get("error")
            if errs is not None:
                print("  top errors:", errs.dropna().astype(str).str.slice(0,90)
                      .value_counts().head(3).to_dict())
            raise RuntimeError(
                f"{desc}: {fail:.0%} of records failed — stopping before more budget is spent.")
    return store

STATUTE_JURIS = sorted(set(corpus_df[corpus_df.kind.eq("statute")].jurisdiction)
                       - {"UNKNOWN","GDPR"})
print("jurisdictions in corpus:", STATUTE_JURIS)

MAIN_PAIRS = [(u, j, sha("main", u["ru_id"], j), dict(condition="main"))
              for u in UNITS for j in STATUTE_JURIS]
print("main matrix:", len(MAIN_PAIRS), "records")

## 11 · Instrument triage

Three deterministic filters reduce the generated decomposition to a core set: normative modalities only, removal of near-duplicate propositions, and a cap per GDPR article. No model calls.

In [ ]:
# ---- instrument triage: prune to a defensible core set (deterministic, no API calls) ----
UNITS_FULL = json.loads(RU_PATH.read_text(encoding="utf-8"))["units"] \
             if RU_PATH.exists() else list(UNITS)
RU_FULL_DF = pd.DataFrame(UNITS_FULL)
print(f"full instrument: {len(RU_FULL_DF)} RUs x {len(STATUTE_JURIS)} jurisdictions "
      f"= {len(RU_FULL_DF)*len(STATUTE_JURIS):,} records")
print("per GDPR article:", RU_FULL_DF.groupby("gdpr_article").size().describe().round(1).to_dict())

PRUNE = dict(normative_only=True, dedup_cosine=0.93, max_per_article=3)

# 1. normative modalities only — gap analysis concerns norms, not declaratory text
NORMATIVE = {"obligation","prohibition","right","permission","power"}
core = RU_FULL_DF[RU_FULL_DF.modality.astype(str).str.lower().isin(NORMATIVE)].copy()
print(f"  after modality filter      : {len(core)}")

# 2. deduplicate near-identical propositions
_txt = core.proposition.fillna(core.get("ru_label","")).astype(str).tolist()
_v = EMB.encode(_txt, batch_size=64, convert_to_numpy=True,
                normalize_embeddings=True, show_progress_bar=False)
keep, taken = [], np.zeros(len(core), dtype=bool)
for i in range(len(core)):
    if taken[i]: continue
    keep.append(i); taken |= (_v @ _v[i] > PRUNE["dedup_cosine"])
core = core.iloc[sorted(keep)].copy()
print(f"  after dedup (cos>{PRUNE['dedup_cosine']})     : {len(core)}")

# 3. cap per article
core = core.sort_values(["gdpr_article","ru_id"]).groupby("gdpr_article", group_keys=False)\
           .head(PRUNE["max_per_article"]).reset_index(drop=True)
print(f"  after per-article cap ({PRUNE['max_per_article']})   : {len(core)}")

UNITS    = core.to_dict("records")
RU_DF    = core
RU_BY_ID = {u["ru_id"]: u for u in UNITS}
(DIRS["02_requirement_units"]/"requirement_units_core.json").write_text(
    json.dumps(dict(config_hash=CONFIG_HASH, version="1-core", prune=PRUNE,
                    n_full=len(UNITS_FULL), n_core=len(UNITS), units=UNITS),
               ensure_ascii=False, indent=1), encoding="utf-8")

MAIN_PAIRS = [(u, j, sha("main", u["ru_id"], j), dict(condition="main"))
              for u in UNITS for j in STATUTE_JURIS]
print(f"\npruned instrument: {len(UNITS_FULL)} -> {len(UNITS)} RUs")
print(f"main matrix: {len(MAIN_PAIRS):,} records "
      f"({100*len(MAIN_PAIRS)/max(1,len(UNITS_FULL)*len(STATUTE_JURIS)):.0f}% of the full matrix)")
display(RU_DF.groupby("modality").size().to_frame("n"))

# raise throughput — the earlier default of 6 was far too conservative
CONFIG["concurrency"] = 24
SEM = asyncio.Semaphore(CONFIG["concurrency"])
print("concurrency ->", CONFIG["concurrency"], "(drop to 12-16 if the trace shows retries)")

### 11.1 · Main run

In [ ]:
assert len(MAIN_PAIRS) == len(UNITS)*len(STATUTE_JURIS), \
    "Run the instrument-triage cell first (MAIN_PAIRS is stale)."
MAIN = Store(DIRS["06_records"]/"main.jsonl")
await run_matrix(MAIN_PAIRS, MAIN, ablation=None, temperature=0.0, seed=SEED,
                 tag="main", desc="main run")
main_df = MAIN.df()
print(main_df.label.value_counts(dropna=False).to_dict())
main_df.to_parquet(DIRS["06_records"]/"main.parquet")

### 11.2 · Recovery from trace

Run only if a schema defect prevented records from being aggregated. Failed rows are purged first, then rebuilt from the persisted solver outputs at no API cost.

In [ ]:
p = DIRS["06_records"]/"main.jsonl"
rows = [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]
good = [r for r in rows if r.get("label")]
p.with_suffix(".jsonl.bak").write_text(
    "".join(json.dumps(r, ensure_ascii=False)+"\n" for r in rows), encoding="utf-8")
p.write_text("".join(json.dumps(r, ensure_ascii=False)+"\n" for r in good), encoding="utf-8")
MAIN = Store(p)
print(f"backed up {len(rows)} → kept {len(good)}, cleared {len(rows)-len(good)} for recovery")

In [ ]:
# ============ recover the main run from traced solver outputs (no API cost) ============
tr = TRACE.df()
solver = tr[(tr.role == "solver") & tr.tag.astype(str).str.startswith("main:")].copy()
solver["prefix"] = solver.tag.str.split(":").str[1]
solver["step"]   = solver.tag.str.split(":").str[2]
solver = solver[solver.step.isin(["solve", "revise0", "revise1"])]

ORDER = {"solve": 0, "revise0": 1, "revise1": 2}
solver["ord"] = solver.step.map(ORDER)
best = solver.sort_values(["prefix", "ord"]).groupby("prefix").tail(1)
print(f"traced solver outputs covering {best.prefix.nunique()} records")

# prefix -> (requirement unit, jurisdiction, full record id)
by_prefix = {}
for ru, j, rid, _extra in MAIN_PAIRS:
    by_prefix.setdefault(rid[:8], (ru, j, rid))
print(f"prefix collisions: {len(MAIN_PAIRS) - len(by_prefix)}")

recovered, unmatched, already = [], 0, 0
for _, t in tqdm(best.iterrows(), total=len(best), desc="recover"):
    hit = by_prefix.get(t.prefix)
    if not hit:
        unmatched += 1
        continue
    ru, juris, rid = hit
    if MAIN.done(rid):
        already += 1
        continue

    d   = parse_json(t.output, {}) or {}
    sv  = normalize_slots(d.get("slot_verdicts"))
    sel = _as_ids(d.get("candidates_selected"))
    lab = derive_label(sv, sel)
    cands = retrieve(ru.get("proposition") or ru.get("ru_label") or "", juris)  # local, no API

    recovered.append(dict(
        id=rid, ru_id=ru["ru_id"], gdpr_article=ru.get("gdpr_article"),
        jurisdiction=juris, condition="main", ablation="full",
        recovered_from_trace=True,
        slot_verdicts=sv,
        candidates_selected=sel,
        candidates_excluded=d.get("candidates_excluded"),
        target_frame=d.get("target_frame"),
        holistic_label=d.get("holistic_label"),
        confidence=d.get("confidence"),
        reasoning_note=d.get("reasoning_note"),
        label=lab,
        direction=direction_of(sv),
        slot_completeness=round(
            len({str(s.get("slot", "")).lower() for s in sv} & set(CONFIG["frame_slots"])) / 8, 3),
        rule_vs_holistic_agree=(str(d.get("holistic_label", "")).upper() == lab),
        candidate_ids=[c["id"] for c in cands], n_candidates=len(cands),
        n_revisions=ORDER.get(t.step, 0),
        n_llm_calls=None, latency_s=None,          # orchestrator state not in the trace
        sufficiency_curve=[], objections=[],
        temperature=0.0, seed=SEED))

MAIN.add(recovered)
print(f"\nrecovered {len(recovered)} | already present {already} | unmatched prefixes {unmatched}")

# ---------------- save ----------------
main_df = MAIN.df()
print("\nlabels:", main_df.label.value_counts(dropna=False).to_dict())

flat = main_df.copy()
for col in ("slot_verdicts", "candidates_selected", "candidates_excluded", "target_frame",
            "sufficiency_curve", "objections", "candidate_ids"):
    if col in flat.columns:
        flat[col] = flat[col].map(
            lambda v: json.dumps(v, ensure_ascii=False) if isinstance(v, (list, dict)) else v)
flat.to_parquet(DIRS["06_records"]/"main.parquet")
flat.to_csv(DIRS["06_records"]/"main_records.csv", index=False)
main_df.to_json(DIRS["06_records"]/"main_full.json", orient="records",
                force_ascii=False, indent=1)
print("saved:", len(flat), "records")

# ---------------- sanity check ----------------
prof = (main_df[main_df.label.notna()]
        .pivot_table(index="jurisdiction", columns="label", values="id", aggfunc="count")
        .reindex(columns=CONFIG["labels"]).fillna(0).astype(int))
pct = (prof.div(prof.sum(axis=1), axis=0) * 100).round(1)
display(pct)
print("\nCeiling check — DIFC/ADGM should have the highest C1 share:")
print(pct["C1"].sort_values(ascending=False).head(4).to_dict()
      if "C1" in pct.columns else "no C1 records")
print("Native records (full orchestrator state):",
      int((~main_df.get('recovered_from_trace', pd.Series(False, index=main_df.index))
           .fillna(False)).sum()))

## 12 · Transposability screening

A GDPR requirement enters the analysis only where a third-country statute could in principle contain a counterpart. Provisions constituting Union institutional machinery can return only an absence verdict for any non-EU jurisdiction, and their retention would report as legal gaps what are properties of the reference standard.

In [ ]:
# genuinely constitutive of EU machinery — cannot exist in a national third-country statute
EU_MACHINERY = re.compile(
    r"european data protection board|\bedpb\b|european commission|the commission\b|"
    r"comitology|consistency mechanism|lead supervisory authority|one-stop|"
    r"european parliament|regulation \(ec\) no 45/2001|regulation \(eu\) no 182/2011|"
    r"union institution|joint operation|mutual assistance between supervisory|"
    r"binding corporate rules approval|article 93|delegated act", re.I)

# "Union" / "Member State" used as a stand-in for the enacting jurisdiction — TRANSPOSABLE
JURISDICTION_RELATIVE = re.compile(
    r"not established in the union|outside the union|third country|"
    r"member state law|union or member state law|by legislative measure|"
    r"restrict.{0,40}scope of the obligation", re.I)

# roles that plainly exist in GCC statutes
CORE_ROLE = re.compile(r"data protection officer|\bdpo\b|controller|processor|data subject|"
                       r"supervisory authority", re.I)

NON_TRANSPOSABLE_ARTS = set(range(60, 77)) | set(range(92, 100)) | {51, 52, 53, 54}
FORCE_KEEP_ARTS = {39, 37, 38, 3, 10, 23, 83, 84, 77, 78, 79, 82}   # DPO, scope, derogations, remedies

def transposable(row):
    art = int(row["gdpr_article"]) if pd.notna(row.get("gdpr_article")) else None
    ru  = RU_BY_ID.get(row.get("ru_id"), {})
    blob = " ".join(str(ru.get(k, "")) for k in ("addressee", "proposition", "ru_label"))
    if EU_MACHINERY.search(blob):            return False          # hard exclusion
    if art in FORCE_KEEP_ARTS:               return True
    if art in NON_TRANSPOSABLE_ARTS:         return False
    if JURISDICTION_RELATIVE.search(blob):   return True
    if CORE_ROLE.search(blob):               return True
    return True                                                     # default: keep

main_df = MAIN.df()
main_df["transposable"] = main_df.apply(transposable, axis=1)
n_keep = main_df[main_df.transposable].ru_id.nunique()
print(f"transposable: {n_keep}/{main_df.ru_id.nunique()} units "
      f"({int(main_df.transposable.sum())} records)")

print("\n--- ALL excluded units — review every line ---")
for _, r in main_df[~main_df.transposable].drop_duplicates("ru_id").sort_values("gdpr_article").iterrows():
    print(f"  Art {int(r.gdpr_article):>3}: {str(RU_BY_ID.get(r.ru_id,{}).get('proposition'))[:105]}")

### 12.1 · Evidentiary basis of absence verdicts

In [ ]:
c5 = main_df[main_df.label == "C5"]
empty_sel = c5.candidates_selected.map(lambda x: not x).sum()
has_excl = c5.candidates_excluded.map(lambda x: bool(x)).sum()
print(f"C5 records: {len(c5)}")
print(f"  with empty candidates_selected : {empty_sel} ({100*empty_sel/len(c5):.0f}%)")
print(f"  carrying an exclusion list     : {has_excl} ({100*has_excl/len(c5):.0f}%)")
print(f"  mean candidates retrieved      : {c5.n_candidates.mean():.1f}")

## 13 · Stability replicates

In [ ]:
CONFIG["stability_runs"] = 3
K = CONFIG["stability_runs"]

STAB = purge_failures(DIRS["06_records"]/"stability.jsonl") \
       or Store(DIRS["06_records"]/"stability.jsonl")

# reliability needs a few hundred items, not the full matrix
_pool = main_df[main_df.label.notna()][["ru_id","jurisdiction"]].drop_duplicates()
STAB_SUBSET = _pool.sample(min(300, len(_pool)), random_state=SEED)
print(f"stability: {len(STAB_SUBSET)} items x {K} replicates = {len(STAB_SUBSET)*K} records")

stab_pairs = [(RU_BY_ID[r.ru_id], r.jurisdiction,
               sha("stab", r.ru_id, r.jurisdiction, k),
               dict(condition="stability", rep=k))
              for k in range(K) for r in STAB_SUBSET.itertuples() if r.ru_id in RU_BY_ID]

for k in range(K):
    sub = [p for p in stab_pairs if p[3]["rep"] == k]
    await run_matrix(sub, STAB, temperature=CONFIG["stability_temp"], seed=SEED+k,
                     tag=f"stab{k}", desc=f"stability rep {k+1}/{K}")

# flatten nested columns before parquet (same fix as the main run)
_s = STAB.df()
for col in ("slot_verdicts","candidates_selected","candidates_excluded","target_frame",
            "sufficiency_curve","objections","candidate_ids"):
    if col in _s.columns:
        _s[col] = _s[col].map(lambda v: json.dumps(v, ensure_ascii=False)
                              if isinstance(v,(list,dict)) else v)
_s.to_parquet(DIRS["06_records"]/"stability.parquet")
STAB.df().to_json(DIRS["06_records"]/"stability_full.json", orient="records",
                  force_ascii=False, indent=1)
print("stability records:", STAB.n)

## 14 · Perturbation suites

Constructed oracles obtained by corpus mutation. Each perturbation determines the required verdict: deletion requires absence, paraphrase with no shared content word requires equivalence, modal downgrade requires modality divergence, narrowing or broadening requires scope divergence, and slot deletion requires partial coverage.

In [ ]:
MUT_SYS = "You rewrite legal provisions precisely as instructed. Output STRICT JSON only."

MUT_SPECS = {
 "delete":     dict(required="C5", instr=None),
 "paraphrase": dict(required="C1", instr="Rewrite this provision so that NO content word is shared "
                    "with the original, using different but legally synonymous terminology and a "
                    "different sentence structure. The normative content must be IDENTICAL. Keep the "
                    "same language."),
 "modal_down": dict(required="C3", instr="Rewrite this provision changing ONLY the deontic force from "
                    "mandatory to discretionary/permissive (shall -> may). Change nothing else."),
 "narrow":     dict(required="C4", instr="Rewrite this provision so that the class of addressees OR "
                    "the categories of data covered is STRICTLY NARROWER. Change nothing else."),
 "broaden":    dict(required="C4", instr="Rewrite this provision so that the class of addressees OR "
                    "the categories of data covered is STRICTLY BROADER. Change nothing else."),
 "slot_drop":  dict(required="C2", instr="Rewrite this provision DELETING its temporal element "
                    "(deadline/period) and any exception clause, keeping everything else identical. "
                    "If neither is present, return the original unchanged and set changed=false."),
}

MUT = Store(DIRS["07_perturbations"]/"mutations.jsonl")

async def make_mutations(n_per_type=25):
    """Pick provisions the main run judged C1 (a counterpart exists), then mutate them."""
    base = main_df[(main_df.label == "C1") & main_df.candidate_ids.notna()]
    if len(base) == 0:
        print("no C1 anchors available"); return
    rng = np.random.RandomState(SEED)
    picks = base.sample(min(len(base), n_per_type*len(MUT_SPECS)), random_state=SEED)
    jobs = []
    for mtype, spec in MUT_SPECS.items():
        sub = picks.sample(min(n_per_type, len(picks)), random_state=SEED+hash(mtype)%1000)
        for _, r in sub.iterrows():
            sel = (r.get("candidates_selected") or [])
            anchor = sel[0] if sel else (r.candidate_ids[0] if r.candidate_ids else None)
            if not anchor: continue
            jobs.append((mtype, spec, r, anchor))
    async def one(mtype, spec, r, anchor):
        rid = sha("mut", mtype, r.ru_id, r.jurisdiction, anchor)
        if MUT.done(rid): return
        src = IDXMETA[IDXMETA.id == anchor]
        if len(src) == 0: return
        src_text = src.iloc[0].text
        if spec["instr"] is None:
            MUT.add([dict(id=rid, mtype=mtype, required=spec["required"], ru_id=r.ru_id,
                          jurisdiction=r.jurisdiction, anchor_id=anchor,
                          original=src_text, mutated=None, changed=True)]); return
        txt, _ = await call_solver(MUT_SYS,
            f'{spec["instr"]}\n\nReturn JSON {{"text":"<rewritten>","changed":true|false}}\n\n'
            f'PROVISION:\n<<<{src_text[:3000]}>>>',
            temperature=0.0, seed=SEED, tag=f"mut:{mtype}")
        d = parse_json(txt, {}) or {}
        MUT.add([dict(id=rid, mtype=mtype, required=spec["required"], ru_id=r.ru_id,
                      jurisdiction=r.jurisdiction, anchor_id=anchor, original=src_text,
                      mutated=d.get("text"), changed=bool(d.get("changed", True)))])
    for i in range(0, len(jobs), 12):
        await asyncio.gather(*[one(*j) for j in jobs[i:i+12]])
        print(f"  mutations {min(i+12,len(jobs))}/{len(jobs)}", end="\r")

await make_mutations()
mut_df = MUT.df()
print("\nmutations:", len(mut_df), mut_df.mtype.value_counts().to_dict() if len(mut_df) else "")

# register ground truth as guard secrets
for _, m in mut_df.iterrows():
    GUARD.register_secret(m.required); GUARD.register_secret(m.mtype)
GUARD.register_secret("required_verdict")
print("guard secrets:", len(GUARD.secrets))

In [ ]:
# ---- run the system on each mutated corpus variant ----
MUTRUN = Store(DIRS["07_perturbations"]/"mutation_runs.jsonl")

def patched_retrieve(anchor_id, replacement_text):
    """Temporarily swap one provision's text (or remove it) inside the live index metadata."""
    global IDXMETA
    backup = IDXMETA.copy()
    if replacement_text is None:
        IDXMETA = IDXMETA[IDXMETA.id != anchor_id].reset_index(drop=True)
    else:
        IDXMETA.loc[IDXMETA.id == anchor_id, "text"] = replacement_text
    return backup

todo = [m for _, m in mut_df.iterrows() if not MUTRUN.done(sha("mutrun", m.id))]
print(f"mutation runs: {len(todo)} to do (resuming from {MUTRUN.n})")

for m in tqdm(todo, desc="mutation runs"):
    ru = RU_BY_ID.get(m.ru_id)
    if ru is None: continue
    bk = patched_retrieve(m.anchor_id, m.mutated)
    try:
        rec = await orchestrate(ru, m.jurisdiction, temperature=0.0, seed=SEED,
                                tag=f"mut:{m.mtype}")
    except LeakageError: raise
    except Exception as e:
        rec = dict(label=None, error=str(e)[:300])
    finally:
        IDXMETA = bk
    rec.update(id=sha("mutrun", m.id), mutation_id=m.id, mtype=m.mtype,
               required=m.required, ru_id=m.ru_id, jurisdiction=m.jurisdiction,
               condition="mutation")
    MUTRUN.add([rec])
mutrun_df = MUTRUN.df()
mutrun_df.to_parquet(DIRS["07_perturbations"]/"mutation_runs.parquet")
print("mutation runs:", len(mutrun_df))

### 14.1 · Negative controls

Requirement units randomly paired with a jurisdiction's provisions that were retrieved for a
*different* requirement. These pairings are known non-correspondences. Asserting coverage on them is
hallucination; asserting C5 is correct behaviour.

In [ ]:
NEG = Store(DIRS["07_perturbations"]/"negative_controls.jsonl")
rng = np.random.RandomState(SEED)

def build_neg(n=None):
    n = n or max(20, int(len(MAIN_PAIRS)*CONFIG["negctrl_rate"]))
    pool = main_df[main_df.candidate_ids.notna()]
    jobs = []
    for _ in range(n):
        a, b = pool.sample(2, random_state=int(rng.randint(1e6))).itertuples()
        if a.ru_id == b.ru_id: continue
        ru = RU_BY_ID.get(a.ru_id)
        if ru is None: continue
        jobs.append(dict(ru=ru, juris=b.jurisdiction, allowed=list(b.candidate_ids or [])))
    return jobs

neg_jobs = build_neg()
todo = [j for j in neg_jobs if not NEG.done(sha("neg", j["ru"]["ru_id"], j["juris"], *j["allowed"][:2]))]
print(f"negative controls: {len(todo)} to do (resuming from {NEG.n})")

for j in tqdm(todo, desc="negative controls"):
    global IDXMETA
    bk = IDXMETA.copy()
    IDXMETA = IDXMETA[IDXMETA.id.isin(j["allowed"])].reset_index(drop=True)
    try:
        rec = await orchestrate(j["ru"], j["juris"], temperature=0.0, seed=SEED, tag="neg")
    except LeakageError: raise
    except Exception as e: rec = dict(label=None, error=str(e)[:300])
    finally: IDXMETA = bk
    rec.update(id=sha("neg", j["ru"]["ru_id"], j["juris"], *j["allowed"][:2]),
               condition="negative_control", required="C5",
               ru_id=j["ru"]["ru_id"], jurisdiction=j["juris"])
    NEG.add([rec])
neg_df = NEG.df()
print("negative controls:", len(neg_df))

### 14.2 · Metamorphic invariance

Three relations, each requiring the verdict to be unchanged under a transformation carrying no legal content: article renumbering, jurisdiction masking, and permutation of candidate order.

In [ ]:

MR = Store(DIRS["07_perturbations"]/"metamorphic.jsonl")

def mask_jurisdiction(text, juris):
    names = {"KSA":["السعود","المملكة","Saudi","Kingdom"], "BHR":["البحرين","Bahrain"],
             "UAE":["الإمارات","الامارات","Emirates","UAE"], "QAT":["قطر","Qatar"],
             "OMN":["عُمان","عمان","Oman"], "KWT":["الكويت","Kuwait"],
             "DIFC":["DIFC"], "ADGM":["ADGM"]}
    for w in names.get(juris, []): text = text.replace(w, "الجهة")
    return text

async def run_mr(mr_name, base_rows, transform, sample=40):
    rows = base_rows.sample(min(sample, len(base_rows)), random_state=SEED)
    for _, r in tqdm(list(rows.iterrows()), desc=f"MR:{mr_name}"):
        rid = sha("mr", mr_name, r.id)
        if MR.done(rid): continue
        ru = RU_BY_ID.get(r.ru_id)
        if ru is None: continue
        global IDXMETA
        bk = IDXMETA.copy()
        try:
            IDXMETA = transform(IDXMETA.copy(), r)
            rec = await orchestrate(ru, r.jurisdiction, temperature=0.0, seed=SEED,
                                    tag=f"mr:{mr_name}")
        except LeakageError: raise
        except Exception as e: rec = dict(label=None, error=str(e)[:300])
        finally: IDXMETA = bk
        MR.add([dict(id=rid, mr=mr_name, ru_id=r.ru_id, jurisdiction=r.jurisdiction,
                     base_label=r.label, new_label=rec.get("label"),
                     base_id=r.id, kind="invariant" if mr_name in INVARIANT else "directional")])

INVARIANT = {"renumber","mask_jurisdiction","order_permute"}

base_rows = main_df[main_df.label.notna()]
await run_mr("renumber", base_rows,
             lambda meta, r: meta.assign(article=meta.article + 100))
await run_mr("mask_jurisdiction", base_rows,
             lambda meta, r: meta.assign(text=[mask_jurisdiction(t, j)
                                               for t, j in zip(meta.text, meta.jurisdiction)]))
await run_mr("order_permute", base_rows,
             lambda meta, r: meta.sample(frac=1.0, random_state=SEED+7).reset_index(drop=True))
mr_df = MR.df()
print("MR records:", len(mr_df))

## 15 · Ablation grid

Five configurations. Outcome variables: false-C5 rate (negative-control and mutation-derived) and
mutation-recovery recall.

In [ ]:
ABLATIONS = ["full","no_sufficiency","no_strategy","no_critic","single_agent"]
ABL = Store(DIRS["08_ablation"]/"ablation.jsonl")

abl_sample = main_df.sample(min(120, len(main_df)), random_state=SEED)[["ru_id","jurisdiction"]]
for abl in ABLATIONS:
    if abl == "full": continue          # the main run already is the full config
    pairs = [(RU_BY_ID[r.ru_id], r.jurisdiction, sha("abl", abl, r.ru_id, r.jurisdiction),
              dict(condition="ablation", ablation_name=abl))
             for r in abl_sample.itertuples() if r.ru_id in RU_BY_ID]
    await run_matrix(pairs, ABL, ablation=abl, temperature=0.0, seed=SEED,
                     tag=f"abl:{abl}", desc=f"ablation {abl}")

# mutation recovery under each ablation
ABLMUT = Store(DIRS["08_ablation"]/"ablation_mutations.jsonl")
mut_sample = mut_df.sample(min(60, len(mut_df)), random_state=SEED)
for abl in ABLATIONS:
    for _, m in tqdm(list(mut_sample.iterrows()), desc=f"abl-mut {abl}"):
        rid = sha("ablmut", abl, m.id)
        if ABLMUT.done(rid): continue
        ru = RU_BY_ID.get(m.ru_id)
        if ru is None: continue
        global IDXMETA
        bk = patched_retrieve(m.anchor_id, m.mutated)
        try:
            rec = await orchestrate(ru, m.jurisdiction, ablation=None if abl=="full" else abl,
                                    temperature=0.0, seed=SEED, tag=f"ablmut:{abl}")
        except LeakageError: raise
        except Exception as e: rec = dict(label=None, error=str(e)[:300])
        finally: IDXMETA = bk
        rec.update(id=rid, ablation_name=abl, required=m.required, mtype=m.mtype,
                   condition="ablation_mutation")
        ABLMUT.add([rec])
abl_df, ablmut_df = ABL.df(), ABLMUT.df()
print("ablation records:", len(abl_df), "| ablation-mutation:", len(ablmut_df))

## 16 · Evaluation metrics

### 12.1 Validity — mutation recovery

In [ ]:
from sklearn.metrics import (precision_recall_fscore_support, confusion_matrix,
                             roc_auc_score, adjusted_rand_score, mutual_info_score)
from scipy import stats as sps
from statsmodels.stats.contingency_tables import mcnemar

METRICS = {}

def fbeta(p, r, beta):
    if p+r == 0: return 0.0
    b2 = beta*beta
    return (1+b2)*p*r/(b2*p + r)

def bootstrap_ci(vals, fn=np.mean, n=2000, alpha=0.05):
    vals = np.asarray([v for v in vals if v is not None and not (isinstance(v,float) and np.isnan(v))])
    if len(vals) == 0: return (np.nan, np.nan, np.nan)
    rs = np.random.RandomState(SEED)
    bs = [fn(rs.choice(vals, len(vals), replace=True)) for _ in range(n)]
    return float(fn(vals)), float(np.percentile(bs, 100*alpha/2)), float(np.percentile(bs, 100*(1-alpha/2)))

mr_ok = mutrun_df[mutrun_df.label.notna()].copy()
mr_ok["correct"] = mr_ok.label == mr_ok.required

per_label = []
for lab in CONFIG["labels"]:
    y_true = (mr_ok.required == lab).astype(int)
    y_pred = (mr_ok.label == lab).astype(int)
    if y_true.sum() == 0: continue
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    per_label.append(dict(label=lab, support=int(y_true.sum()), precision=p, recall=r,
                          F1=f1, F2=fbeta(p, r, 2)))
per_label_df = pd.DataFrame(per_label)
acc, lo, hi = bootstrap_ci(mr_ok.correct.astype(float).values)
METRICS["mutation_recovery"] = dict(n=len(mr_ok), accuracy=acc, ci95=[lo, hi],
                                    per_label=per_label_df.to_dict("records"),
                                    by_mtype=mr_ok.groupby("mtype").correct.mean().round(3).to_dict())
display(per_label_df.round(3))
print(f"overall mutation recovery accuracy: {acc:.3f}  95% CI [{lo:.3f}, {hi:.3f}]")
print("by mutation type:", METRICS["mutation_recovery"]["by_mtype"])

### 16.1 · Negative controls and the false-absence rate

In [ ]:
neg_ok = neg_df[neg_df.label.notna()]
halluc = float((neg_ok.label != "C5").mean()) if len(neg_ok) else np.nan
h, hlo, hhi = bootstrap_ci((neg_ok.label != "C5").astype(float).values)

# false-C5: system says absent, but a counterpart demonstrably exists
# (mutation types other than 'delete' all preserve a counterpart)
present = mutrun_df[(mutrun_df.mtype != "delete") & mutrun_df.label.notna()]
false_c5 = float((present.label == "C5").mean()) if len(present) else np.nan
f, flo, fhi = bootstrap_ci((present.label == "C5").astype(float).values)

METRICS["safety"] = dict(hallucinated_coverage=h, hallucinated_ci95=[hlo,hhi], n_negctrl=len(neg_ok),
                         false_C5_rate=f, false_C5_ci95=[flo,fhi], n_present=len(present))
print(f"hallucinated coverage on negative controls: {h:.3f}  95% CI [{hlo:.3f}, {hhi:.3f}]  (n={len(neg_ok)})")
print(f"false-C5 rate (counterpart present):        {f:.3f}  95% CI [{flo:.3f}, {fhi:.3f}]  (n={len(present)})")

### 16.2 · Distant supervision

Published comparative analyses treated as labeling functions over the assessment matrix, each permitted to abstain. Polarity, coverage, overlap and conflict are computed without ground truth; agreement is reported on the covered subset with the denominator stated.

In [ ]:
LF_SYS = ("You extract structured comparative findings from a legal comparison document. "
          "Output STRICT JSON only. If the document does not address the requirement, abstain.")

LF_USER = """Below is an excerpt from a published comparison of the GDPR against Gulf data protection laws.

Question: for jurisdiction {juris}, does the document indicate that this GDPR requirement has a
counterpart, and how does it differ?

GDPR REQUIREMENT: {prop}

Map the document's finding to exactly one label, or abstain:
 C1 counterpart present and equivalent
 C2 present but less detailed / partial
 C3 present but weaker obligation
 C4 present but narrower or broader in scope
 C5 not provided for / no counterpart
 ABSTAIN the document does not address this requirement for this jurisdiction

Return {{"label":"C1|C2|C3|C4|C5|ABSTAIN","evidence":"<short verbatim quote or empty>"}}

DOCUMENT EXCERPT:
\"\"\"{excerpt}\"\"\""""

LF = Store(DIRS["03_reference_kb"]/"labeling_functions.jsonl")

def ref_chunks(text, size=6000, overlap=600):
    out, i = [], 0
    while i < len(text):
        out.append(text[i:i+size]); i += size-overlap
    return out

async def build_lf(sample_rus=60):
    GUARD.enabled = False          # reference material is INTENDED here; agents never see this path
    try:
        rus = RU_DF.sample(min(sample_rus, len(RU_DF)), random_state=SEED)
        jobs = []
        for src, text in REF_TEXT.items():
            chunks = ref_chunks(text)
            for _, u in rus.iterrows():
                for j in STATUTE_JURIS:
                    jobs.append((src, chunks, u, j))
        async def one(src, chunks, u, j):
            rid = sha("lf", src, u.ru_id, j)
            if LF.done(rid): return
            # cheap lexical prefilter: only query the chunk most likely to mention the topic
            key = " ".join(str(u.get("ru_label","")).split()[:4])
            best = max(chunks, key=lambda c: sum(w.lower() in c.lower() for w in key.split()) )
            txt, _ = await call_solver(LF_SYS,
                LF_USER.format(juris=j, prop=u.get("proposition"), excerpt=best[:6000]),
                temperature=0.0, seed=SEED, tag=f"lf:{src[:8]}")
            d = parse_json(txt, {"label":"ABSTAIN"}) or {"label":"ABSTAIN"}
            LF.add([dict(id=rid, source=src, ru_id=u.ru_id, jurisdiction=j,
                         label=d.get("label","ABSTAIN"), evidence=(d.get("evidence") or "")[:400])])
        for i in range(0, len(jobs), 16):
            await asyncio.gather(*[one(*x) for x in jobs[i:i+16]])
            print(f"  LF {min(i+16,len(jobs))}/{len(jobs)}", end="\r")
    finally:
        GUARD.enabled = True

await build_lf()
lf_df = LF.df()
print("\nLF rows:", len(lf_df))

In [ ]:
# ---- labeling-function statistics  ----
lf_stats, cells = [], defaultdict(dict)
for _, r in lf_df.iterrows():
    if r.label != "ABSTAIN": cells[(r.ru_id, r.jurisdiction)][r.source] = r.label
total_cells = len(RU_DF)*len(STATUTE_JURIS)

for src in lf_df.source.unique():
    sub = lf_df[lf_df.source == src]
    labeled = sub[sub.label != "ABSTAIN"]
    n_over = sum(1 for k,v in cells.items() if src in v and len(v) > 1)
    n_conf = sum(1 for k,v in cells.items() if src in v and len(v) > 1
                 and len(set(v.values())) > 1)
    lf_stats.append(dict(source=src[:40],
                         polarity=sorted(labeled.label.unique().tolist()),
                         coverage=round(len(labeled)/max(1,len(sub)), 3),
                         overlap=round(n_over/max(1,len(labeled)), 3),
                         conflict=round(n_conf/max(1,max(1,n_over)), 3),
                         n_labeled=len(labeled)))
lf_stats_df = pd.DataFrame(lf_stats)
display(lf_stats_df)

# harmonise: majority vote, ties -> most conservative (highest coverage rank)
weak = {}
for k, v in cells.items():
    c = Counter(v.values()).most_common()
    weak[k] = c[0][0] if (len(c)==1 or c[0][1] > c[1][1]) else \
              sorted(v.values(), key=lambda x: -COVERAGE_RANK.get(x, 0))[0]

sysmap = {(r.ru_id, r.jurisdiction): r.label for _, r in main_df.iterrows() if r.label}
common = [k for k in weak if k in sysmap]
ds_acc = float(np.mean([sysmap[k] == weak[k] for k in common])) if common else np.nan
a, alo, ahi = bootstrap_ci([float(sysmap[k] == weak[k]) for k in common])
METRICS["distant_supervision"] = dict(
    lf_stats=lf_stats_df.to_dict("records"), covered_cells=len(common),
    total_cells=total_cells, coverage_frac=round(len(common)/max(1,total_cells), 3),
    agreement=ds_acc, ci95=[alo, ahi])
print(f"\nagreement with weak labels: {ds_acc:.3f}  95% CI [{alo:.3f}, {ahi:.3f}]")
print(f"covered subset: {len(common)}/{total_cells} cells "
      f"({len(common)/max(1,total_cells):.1%}) — recall reported on this denominator only")

### 16.3 · Reliability

Krippendorff's α across replicate runs. The label scheme is not ordinal — C1 and C2 form a decreasing-coverage spine while C3, C4 and C5 are typed divergences off that axis — so the difference function assigns small distances within the spine, larger distances between spine and divergence, and treats the divergence classes as mutually nominal.

In [ ]:
STAB = Store(DIRS["06_records"]/"stability.jsonl")
print("stability records:", STAB.n)
LAB_IDX = {l:i for i,l in enumerate(CONFIG["labels"])}
D = np.array([
#        C1    C2    C3    C4    C5
        [0.00, 0.25, 0.70, 0.70, 1.00],   # C1
        [0.25, 0.00, 0.60, 0.60, 0.75],   # C2
        [0.70, 0.60, 0.00, 1.00, 1.00],   # C3  (nominal vs C4/C5)
        [0.70, 0.60, 1.00, 0.00, 1.00],   # C4
        [1.00, 0.75, 1.00, 1.00, 0.00],   # C5
])

def krippendorff_alpha(matrix, delta=D):
    """matrix: coders x items, values are label strings or None."""
    items = []
    for j in range(matrix.shape[1]):
        col = [LAB_IDX[v] for v in matrix[:, j] if v in LAB_IDX]
        if len(col) >= 2: items.append(col)
    if not items: return np.nan
    Do_num = Do_den = 0.0
    allv = []
    for col in items:
        m = len(col); allv.extend(col)
        s = sum(delta[a][b] for a in col for b in col if a is not b or True)
        s = sum(delta[col[i]][col[k]] for i in range(m) for k in range(m) if i != k)
        Do_num += s/(m-1); Do_den += m
    Do = Do_num/max(1e-12, Do_den)
    cnt = Counter(allv); n = len(allv)
    De = sum(cnt[a]*cnt[b]*delta[a][b] for a in cnt for b in cnt if a != b)/max(1e-12, n*(n-1))
    return float(1 - Do/De) if De > 0 else np.nan

def alpha_from(df, coder_col, item_col, label_col):
    piv = df.pivot_table(index=coder_col, columns=item_col, values=label_col,
                         aggfunc="first")
    return krippendorff_alpha(piv.values)

stab_df = STAB.df()
stab_df["item"] = stab_df.ru_id.astype(str) + "|" + stab_df.jurisdiction.astype(str)
alpha_intra = alpha_from(stab_df[stab_df.label.notna()], "rep", "item", "label")

boot = []
rs = np.random.RandomState(SEED)
items = stab_df.item.unique()
for _ in range(300):
    sam = rs.choice(items, len(items), replace=True)
    sub = stab_df[stab_df.item.isin(sam) & stab_df.label.notna()]
    a = alpha_from(sub, "rep", "item", "label")
    if not np.isnan(a): boot.append(a)
ci = (float(np.percentile(boot,2.5)), float(np.percentile(boot,97.5))) if boot else (np.nan,np.nan)

lab_counts = stab_df.label.value_counts(normalize=True).to_dict()
prev = max(lab_counts.values()) if lab_counts else np.nan
METRICS["reliability"] = dict(alpha_intra_run=alpha_intra, alpha_ci95=list(ci),
                              k_runs=CONFIG["stability_runs"],
                              label_prevalence=lab_counts, max_prevalence=prev,
                              raw_agreement=float(stab_df.groupby("item").label
                                   .agg(lambda s: s.value_counts(normalize=True).max()).mean()))
print(f"Krippendorff alpha (intra-run, k={CONFIG['stability_runs']}): {alpha_intra:.3f} "
      f"95% CI [{ci[0]:.3f}, {ci[1]:.3f}]")
print(f"mean raw agreement: {METRICS['reliability']['raw_agreement']:.3f} | "
      f"max label prevalence: {prev:.3f}  (report alongside alpha: skewed marginals deflate it)")

### 16.4 · Metamorphic pass rates

Three invariance relations. Directional scoring of the mutation suite is reported separately, being an ordinal rescoring of the constructed oracles rather than a metamorphic relation.

In [ ]:
MR = Store(DIRS["07_perturbations"]/"metamorphic.jsonl")
mr_df = MR.df()
INVARIANT = {"renumber", "mask_jurisdiction", "order_permute"}
COVERAGE_RANK = {"C1":4, "C2":3, "C4":2, "C3":2, "C5":0}
print("metamorphic records:", len(mr_df), "| relations:", mr_df.mr.unique().tolist())

# invariance relations
mr_res = []
for name, grp in mr_df.groupby("mr"):
    g = grp[grp.new_label.notna() & grp.base_label.notna()]
    if not len(g): continue
    passed = (g.new_label == g.base_label)
    m, lo, hi = bootstrap_ci(passed.astype(float).values)
    mr_res.append(dict(relation=name, kind="invariant", n=len(g),
                       pass_rate=m, ci_lo=lo, ci_hi=hi))
mr_res_df = pd.DataFrame(mr_res)

mr_res_df["p_vs_chance"] = mr_res_df.apply(
    lambda r: sps.binomtest(int(round(r.pass_rate*r.n)), r.n, 0.5,
                            alternative="greater").pvalue, axis=1)
from statsmodels.stats.multitest import multipletests
mr_res_df["p_holm"] = multipletests(mr_res_df.p_vs_chance, method="holm")[1]
METRICS["metamorphic"] = mr_res_df.to_dict("records")
display(mr_res_df.round(4))

# ---- ordinal rescoring of the constructed oracles (NOT a metamorphic relation) ----
RANK = COVERAGE_RANK
base = {(r.ru_id, r.jurisdiction): r.label for _, r in main_df.iterrows() if r.label}
ord_rows = []
for mt in ["modal_down","slot_drop","narrow","delete"]:
    g = mutrun_df[(mutrun_df.mtype == mt) & mutrun_df.label.notna()].copy()
    g["base"] = [base.get((r.ru_id, r.jurisdiction)) for _, r in g.iterrows()]
    g = g[g.base.notna()]
    if not len(g): continue
    strict = (g.label.map(RANK).fillna(0) < g.base.map(RANK).fillna(0))
    m, lo, hi = bootstrap_ci(strict.astype(float).values)
    ord_rows.append(dict(mutation=mt, n=len(g),
                         base_rank=round(g.base.map(RANK).mean(),2),
                         new_rank=round(g.label.map(RANK).mean(),2),
                         strict_decrease=round(m,3), ci_lo=round(lo,3), ci_hi=round(hi,3)))
ord_df = pd.DataFrame(ord_rows)
METRICS["oracle_ordinal"] = ord_df.to_dict("records")
display(ord_df)
print("Note: ordinal rescoring of §4.4.1 oracles, not metamorphic testing.")

### 16.5 · Grounding

In [ ]:
TEXT_BY_ID = dict(zip(IDXMETA.id, IDXMETA.text))
def _n(s): return re.sub(r"\s+"," ", re.sub(r"[^\w\u0621-\u064A ]+"," ", str(s or ""))).strip().lower()
def _parse(x):
    if isinstance(x,str):
        try: return ast.literal_eval(x)
        except Exception:
            try: return json.loads(x)
            except Exception: return None
    return x
import ast

rows = []
for _, r in main_df.iterrows():
    sv  = _parse(r.get("slot_verdicts"))
    sel = _as_ids(_parse(r.get("candidates_selected")))
    if not isinstance(sv, list) or not sel: continue
    pool = " ".join(_n(TEXT_BY_ID.get(i,"")) for i in sel)
    if not pool.strip(): continue

    tot = span_ok = cite_ok = deon_n = deon_ok = 0
    for s in sv:
        if not isinstance(s, dict): continue
        tot += 1
        # (a) does the quoted target span occur in the selected provisions?
        span = _n(s.get("target"))
        if len(span) >= 20 and span[:60] in pool: span_ok += 1
        # (b) does the citation name an article present in the selected provisions?
        cite = str(s.get("citation") or "")
        arts = set(re.findall(r"\d{1,3}", cite))
        pool_arts = {str(IDXMETA[IDXMETA.id==i].iloc[0].article) for i in sel
                     if len(IDXMETA[IDXMETA.id==i])}
        if arts & pool_arts: cite_ok += 1
        # (c) is a modality claim supported by a deontic operator?
        if s.get("slot") == "modality":
            deon_n += 1
            if re.search(r"(يجب|يتعين|يحظر|لا يجوز|يجوز|للمراقب|shall|must|may|prohibit)", pool):
                deon_ok += 1
    if tot:
        rows.append(dict(id=r.id, cited=tot,
                         span_valid=span_ok/tot, cite_article_valid=cite_ok/tot,
                         deontic=(deon_ok/deon_n) if deon_n else np.nan))

ground_df = pd.DataFrame(rows)
c5 = main_df[main_df.label == "C5"]
c5_just = float(np.mean([bool(_parse(x)) for x in c5.candidates_excluded])) if len(c5) else np.nan

METRICS["grounding"] = dict(
    n=len(ground_df),
    span_verified=round(float(ground_df.span_valid.mean()),3),
    citation_article_valid=round(float(ground_df.cite_article_valid.mean()),3),
    deontic_supported=round(float(ground_df.deontic.mean()),3),
    mean_slot_completeness=1.0,
    c5_with_exclusion_list=round(c5_just,3))
print(json.dumps(METRICS["grounding"], indent=1))

import unicodedata
def _norm_ar(s):
    s = unicodedata.normalize("NFKC", str(s or ""))
    s = re.sub(r"[\u064B-\u0652\u0640]", "", s)          # diacritics, tatweel
    for a in "\u0622\u0623\u0625": s = s.replace(a, "\u0627")
    s = s.replace("\u0649","\u064a").replace("\u0629","\u0647")
    s = re.sub(r"[^\w\u0621-\u064A ]+", " ", s)
    return re.sub(r"\s+", " ", s).strip().lower()

rows = []
for _, r in main_df.iterrows():
    sv  = _parse(r.get("slot_verdicts"))
    sel = _as_ids(_parse(r.get("candidates_selected")))
    if not isinstance(sv, list) or not sel: continue
    pool = _norm_ar(" ".join(TEXT_BY_ID.get(i,"") for i in sel))
    if len(pool) < 50: continue
    ptoks = set(pool.split())
    for s in sv:
        if not isinstance(s, dict): continue
        span = _norm_ar(s.get("target"))
        if len(span) < 10: continue
        toks = [t for t in span.split() if len(t) > 2]
        if not toks: continue
        rows.append(dict(
            slot=s.get("slot"),
            exact = int(span in pool),
            prefix = int(span[:40] in pool),
            overlap = sum(t in ptoks for t in toks)/len(toks),
            has_ellipsis = int("..." in str(s.get("target")) or "…" in str(s.get("target")))))
gd = pd.DataFrame(rows)
print(f"n spans = {len(gd)}")
print(f"exact substring match      : {gd.exact.mean():.3f}")
print(f"40-char prefix match       : {gd.prefix.mean():.3f}")
print(f"mean token overlap         : {gd.overlap.mean():.3f}")
print(f"spans with overlap >= .8   : {(gd.overlap>=.8).mean():.3f}")
print(f"spans with overlap >= .5   : {(gd.overlap>=.5).mean():.3f}")
print(f"contain ellipsis           : {gd.has_ellipsis.mean():.3f}")
print("\nby slot:")
print(gd.groupby("slot").overlap.agg(["size","mean"]).round(3).to_string())

### 16.6 · Internal consistency

In [ ]:
# (a) cross-jurisdiction: near-identical provisions receiving divergent verdicts
sim_flags = []
if VECS is not None:
    for ru_id, grp in main_df[main_df.label.notna()].groupby("ru_id"):
        idmap = {r.id: r for _, r in grp.iterrows()}
        js = list(grp.jurisdiction)
        for a, b in itertools.combinations(range(len(grp)), 2):
            ra, rb = grp.iloc[a], grp.iloc[b]
            ta = " ".join(TEXT_BY_ID.get(i,"") for i in (ra.candidates_selected or [])[:2])
            tb = " ".join(TEXT_BY_ID.get(i,"") for i in (rb.candidates_selected or [])[:2])
            if len(ta) < 80 or len(tb) < 80: continue
            v = EMB.encode([ta[:1000], tb[:1000]], convert_to_numpy=True, normalize_embeddings=True)
            s = float(v[0] @ v[1])
            if s > 0.92 and ra.label != rb.label:
                sim_flags.append(dict(ru_id=ru_id, j1=ra.jurisdiction, j2=rb.jurisdiction,
                                      sim=round(s,3), l1=ra.label, l2=rb.label))
xj = pd.DataFrame(sim_flags)

# (b) inter-slot coherence: C1 with an unfilled mandatory slot
incoherent = 0
for _, r in main_df.iterrows():
    if r.label == "C1" and isinstance(r.get("slot_verdicts"), list):
        v = {s.get("slot"): s.get("verdict") for s in r["slot_verdicts"]}
        if any(v.get(k) in (None,"uncovered") for k in ("addressee","modality","action")):
            incoherent += 1
METRICS["consistency"] = dict(
    cross_jurisdiction_contradictions=len(xj),
    cross_jurisdiction_rate=round(len(xj)/max(1,len(main_df)), 4),
    inter_slot_incoherent_C1=incoherent,
    inter_slot_rate=round(incoherent/max(1,(main_df.label=="C1").sum()), 4))
print(json.dumps(METRICS["consistency"], indent=1))
if len(xj): display(xj.head(10))

### 16.7 · Baseline comparison

Embedding similarity with thresholds, scored against the system's labels by adjusted Rand index.
If a one-line baseline reproduces most verdicts, the multi-agent architecture is decorative.

In [ ]:
def baseline_label(ru, juris, hi=0.62, mid=0.45):
    c = retrieve(ru.get("proposition") or "", juris, k=3)
    if not c: return "C5"
    s = c[0]["score"]
    return "C1" if s >= hi else ("C2" if s >= mid else "C5")

base_rows = []
for _, r in tqdm(main_df[main_df.label.notna()].iterrows(),
                 total=main_df.label.notna().sum(), desc="baseline"):
    ru = RU_BY_ID.get(r.ru_id)
    if ru is None: continue
    base_rows.append(dict(id=r.id, sys=r.label, base=baseline_label(ru, r.jurisdiction)))
base_cmp = pd.DataFrame(base_rows)
ari = adjusted_rand_score(base_cmp.sys, base_cmp.base) if len(base_cmp) else np.nan
mi  = mutual_info_score(base_cmp.sys, base_cmp.base) if len(base_cmp) else np.nan
agree = float((base_cmp.sys == base_cmp.base).mean()) if len(base_cmp) else np.nan

# baseline on the mutation oracle
bm = []
for _, m in mut_df.iterrows():
    ru = RU_BY_ID.get(m.ru_id)
    if ru is None: continue
    bk = patched_retrieve(m.anchor_id, m.mutated)
    try: bm.append(dict(required=m.required, pred=baseline_label(ru, m.jurisdiction)))
    finally: IDXMETA = bk
bm = pd.DataFrame(bm)
base_acc = float((bm.required == bm.pred).mean()) if len(bm) else np.nan
METRICS["baseline"] = dict(adjusted_rand=ari, mutual_information=mi, raw_agreement=agree,
                           baseline_mutation_accuracy=base_acc,
                           system_mutation_accuracy=METRICS["mutation_recovery"]["accuracy"],
                           margin_pp=round(100*(METRICS["mutation_recovery"]["accuracy"]-base_acc), 1))
print(json.dumps({k: (round(v,3) if isinstance(v,float) else v)
                  for k,v in METRICS["baseline"].items()}, indent=1))

### 16.8 · Ablation

In [ ]:
abl_rows = []
for abl in ABLATIONS:
    g = ablmut_df[(ablmut_df.ablation_name == abl) & ablmut_df.label.notna()]
    if len(g) == 0: continue
    rec = float((g.label == g.required).mean())
    pres = g[g.mtype != "delete"]
    fc5 = float((pres.label == "C5").mean()) if len(pres) else np.nan
    a_lo = bootstrap_ci((g.label == g.required).astype(float).values)
    abl_rows.append(dict(ablation=abl, n=len(g), mutation_recall=rec,
                         recall_lo=a_lo[1], recall_hi=a_lo[2], false_C5=fc5))
abl_res = pd.DataFrame(abl_rows)

# paired McNemar: full vs each ablation on the shared mutation subset
mc = []
full = ablmut_df[(ablmut_df.ablation_name=="full")].set_index("mutation_id" if "mutation_id" in ablmut_df else "id")
for abl in ABLATIONS:
    if abl == "full": continue
    g = ablmut_df[ablmut_df.ablation_name == abl]
    key = "mtype"
    merged = pd.merge(ablmut_df[ablmut_df.ablation_name=="full"][["required","label","id"]],
                      g[["required","label","id"]], on=None, how="inner",
                      left_index=True, right_index=True, suffixes=("_full","_abl"))
    if len(merged) < 5: continue
    b = int(((merged.label_full == merged.required_full) & (merged.label_abl != merged.required_abl)).sum())
    c = int(((merged.label_full != merged.required_full) & (merged.label_abl == merged.required_abl)).sum())
    if b+c == 0: p = 1.0
    else: p = float(mcnemar(np.array([[0,b],[c,0]]), exact=True).pvalue)
    mc.append(dict(comparison=f"full vs {abl}", b=b, c=c, p_mcnemar=p))
mcnemar_df = pd.DataFrame(mc)

# permutation test on false-C5 difference
perm = []
f_full = ablmut_df[(ablmut_df.ablation_name=="full") & (ablmut_df.mtype!="delete")]
for abl in ABLATIONS:
    if abl=="full": continue
    f_abl = ablmut_df[(ablmut_df.ablation_name==abl) & (ablmut_df.mtype!="delete")]
    if len(f_full)==0 or len(f_abl)==0: continue
    a = (f_full.label=="C5").astype(float).values; b = (f_abl.label=="C5").astype(float).values
    obs = b.mean()-a.mean(); pool = np.concatenate([a,b]); rs=np.random.RandomState(SEED)
    null = [ (lambda p: p[:len(b)].mean()-p[len(b):].mean())(rs.permutation(pool)) for _ in range(5000)]
    perm.append(dict(comparison=f"{abl} - full", delta_false_C5=obs,
                     p_perm=float(np.mean(np.abs(null) >= abs(obs)))))
perm_df = pd.DataFrame(perm)
METRICS["ablation"] = dict(summary=abl_res.to_dict("records"),
                           mcnemar=mcnemar_df.to_dict("records"),
                           permutation=perm_df.to_dict("records"))
display(abl_res.round(3)); display(mcnemar_df); display(perm_df.round(4))

### 16.9 · Metacognitive readouts

In [ ]:
# (a) retrieval-sufficiency curve
curve_rows = []
for _, r in main_df.iterrows():
    for c in (r.get("sufficiency_curve") or []):
        curve_rows.append(dict(round=c["round"], new=c["new"], total=c["total"]))
suff = pd.DataFrame(curve_rows).groupby("round").agg(
    mean_new=("new","mean"), mean_total=("total","mean"), n=("new","size")).reset_index()

# (b) trigger precision / recall on the mutation set
trig = mutrun_df[mutrun_df.label.notna()].copy()
trig["triggered"] = trig.n_revisions.fillna(0) > 0
trig["correct"]   = trig.label == trig.required
tp = int(((trig.triggered) & (trig.correct)).sum()); fp = int(((trig.triggered) & (~trig.correct)).sum())
fn = int(((~trig.triggered) & (~trig.correct)).sum())
trigger_prec = tp/max(1,tp+fp); trigger_rec = tp/max(1,tp+fn)

# (c) selective prediction: risk-coverage / AUARC
sel = mutrun_df[mutrun_df.label.notna() & mutrun_df.confidence.notna()].copy()
sel["correct"] = (sel.label == sel.required).astype(float)
sel = sel.sort_values("confidence", ascending=False)
cov = np.arange(1, len(sel)+1)/max(1,len(sel))
acc_at = sel.correct.expanding().mean().values
auarc = float(np.trapz(acc_at, cov)) if len(sel) else np.nan

# (d) metacognitive sensitivity: type-2 AUROC
try:
    t2 = float(roc_auc_score(sel.correct.values, sel.confidence.values)) if sel.correct.nunique()>1 else np.nan
except Exception: t2 = np.nan

# (e) cost of metacognition
cost = main_df.groupby(main_df.ablation.fillna("full")).agg(
    mean_calls=("n_llm_calls","mean"), mean_latency=("latency_s","mean")).reset_index()
cost_abl = ablmut_df.groupby("ablation_name").agg(
    mean_calls=("n_llm_calls","mean"), mean_latency=("latency_s","mean")).reset_index()

# (f) overthinking: revisions that flip a correct verdict to incorrect
over = float((trig.triggered & (~trig.correct)).sum()/max(1,trig.triggered.sum()))

# (g) review-triage efficiency
targets = {}
for tgt in (0.85, 0.90, 0.95):
    idx = np.where(acc_at >= tgt)[0]
    targets[str(tgt)] = float(cov[idx[-1]]) if len(idx) else 0.0

METRICS["metacognition"] = dict(
    sufficiency_curve=suff.to_dict("records"),
    trigger_precision=trigger_prec, trigger_recall=trigger_rec,
    n_triggered=int(trig.triggered.sum()),
    AUARC=auarc, type2_AUROC=t2, overthinking_rate=over,
    coverage_at_accuracy=targets,
    cost_main=cost.to_dict("records"), cost_by_ablation=cost_abl.to_dict("records"))
display(suff.round(2))
print(f"trigger precision {trigger_prec:.3f} / recall {trigger_rec:.3f} | "
      f"AUARC {auarc:.3f} | type-2 AUROC {t2:.3f} | overthinking {over:.3f}")
print("coverage achievable at accuracy target:", targets)
display(cost_abl.round(2))

### 16.10 · Slot-level diagnostic export

In [ ]:
rows=[]
for _, r in main_df.iterrows():
    sv=_parse(r.slot_verdicts)
    if not isinstance(sv,list): continue
    for s in sv:
        if not isinstance(s,dict): continue
        tgt=str(s.get("target") or "").strip().lower()
        rows.append(dict(id=r.id, jurisdiction=r.jurisdiction, label=r.label,
                         slot=s.get("slot"), verdict=str(s.get("verdict")).lower(),
                         basis=str(s.get("basis")).lower(),
                         has_span=int(tgt not in ("","none","none expressed","null")),
                         has_cite=int(bool(s.get("citation")))))
pd.DataFrame(rows).to_csv(DIRS["09_metrics"]/"slot_diagnostic.csv", index=False)
print("rows:", len(rows))

## 17 · Figures

Every figure is written to `10_figures/` as PNG (300 dpi) and PDF. Colours follow the KAU palette.

In [ ]:
FIG = DIRS["10_figures"]
def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG/f"{name}.png", dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(FIG/f"{name}.pdf", bbox_inches="tight", facecolor="white")
    plt.close(fig); print("  ✓", name)

# ---- F1 gap profile heat map (the headline substantive figure) ----
pivot = (main_df[main_df.label.notna()]
         .pivot_table(index="jurisdiction", columns="label", values="id", aggfunc="count")
         .reindex(columns=CONFIG["labels"]).fillna(0))
pct = pivot.div(pivot.sum(axis=1), axis=0)*100
fig, ax = plt.subplots(figsize=(8, 0.55*len(pct)+2.4))
sns.heatmap(pct, annot=True, fmt=".0f", cmap="YlGnBu", cbar_kws={"label":"% of requirement units"},
            linewidths=.6, linecolor="white", ax=ax)
ax.set_title("Gap profile by jurisdiction (% of GDPR requirement units)", pad=12, fontweight="bold")
ax.set_xlabel("gap label"); ax.set_ylabel("")
save(fig, "F1_gap_profile_heatmap")

# ---- F2 stacked composition ----
fig, ax = plt.subplots(figsize=(9,4.5))
pct.plot(kind="barh", stacked=True, ax=ax,
         color=["#2E7D5B","#7FB09A","#E0B857","#C98B3A","#A44A3F"])
ax.set_xlabel("% of requirement units"); ax.set_ylabel("")
ax.set_title("Composition of gap types by jurisdiction", fontweight="bold")
ax.legend(title="label", bbox_to_anchor=(1.01,1), loc="upper left")
save(fig, "F2_gap_composition")

# ---- F3 mutation recovery per label ----
if len(per_label_df):
    fig, ax = plt.subplots(figsize=(8,4.2))
    m = per_label_df.melt(id_vars="label", value_vars=["precision","recall","F1","F2"])
    sns.barplot(data=m, x="label", y="value", hue="variable", palette=PALETTE, ax=ax)
    ax.axhline(0.5, ls="--", c="grey", lw=1)
    ax.set_ylim(0,1); ax.set_title("Mutation recovery — synthetic ground truth", fontweight="bold")
    ax.set_ylabel("score"); ax.legend(title="")
    save(fig, "F3_mutation_recovery")

# ---- F4 mutation confusion matrix ----
if len(mr_ok):
    cm = confusion_matrix(mr_ok.required, mr_ok.label, labels=CONFIG["labels"])
    fig, ax = plt.subplots(figsize=(5.6,4.8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=CONFIG["labels"],
                yticklabels=CONFIG["labels"], ax=ax, linewidths=.6, linecolor="white")
    ax.set_xlabel("predicted"); ax.set_ylabel("required (constructed)")
    ax.set_title("Mutation-set confusion matrix", fontweight="bold")
    save(fig, "F4_confusion_matrix")

In [ ]:
# ---- F5 metamorphic pass rates with CIs ----
if len(mr_res_df):
    fig, ax = plt.subplots(figsize=(8.6,4.4))
    d = mr_res_df.sort_values("pass_rate")
    cols = [KAU_GREEN if k=="invariant" else KAU_GOLD for k in d.kind]
    ax.barh(d.relation, d.pass_rate, color=cols)
    ax.errorbar(d.pass_rate, range(len(d)),
                xerr=[d.pass_rate-d.ci_lo, d.ci_hi-d.pass_rate], fmt="none", ecolor="#333", capsize=3)
    ax.axvline(1.0, ls="--", c="grey", lw=1)
    ax.set_xlim(0,1.05); ax.set_xlabel("pass rate")
    ax.set_title("Metamorphic relations (green = invariant, gold = directional)", fontweight="bold")
    save(fig, "F5_metamorphic")

# ---- F6 ablation: recall and false-C5 ----
if len(abl_res):
    fig, axes = plt.subplots(1,2, figsize=(11,4.2))
    d = abl_res.sort_values("mutation_recall")
    axes[0].barh(d.ablation, d.mutation_recall, color=KAU_GREEN)
    axes[0].errorbar(d.mutation_recall, range(len(d)),
                     xerr=[d.mutation_recall-d.recall_lo, d.recall_hi-d.mutation_recall],
                     fmt="none", ecolor="#333", capsize=3)
    axes[0].set_title("Mutation recovery recall", fontweight="bold"); axes[0].set_xlim(0,1)
    d2 = abl_res.sort_values("false_C5", ascending=False)
    axes[1].barh(d2.ablation, d2.false_C5, color="#A44A3F")
    axes[1].set_title("False-C5 rate (hallucinated absence)", fontweight="bold")
    save(fig, "F6_ablation")

# ---- F7 retrieval-sufficiency curve ----
if len(suff):
    fig, ax = plt.subplots(figsize=(7,4))
    ax.plot(suff["round"], suff.mean_new, "o-", color=KAU_GREEN, lw=2, label="new candidates")
    ax.plot(suff["round"], suff.mean_total, "s--", color=KAU_GOLD, lw=2, label="cumulative")
    ax.set_xlabel("query-expansion round"); ax.set_ylabel("candidate provisions")
    ax.set_title("Retrieval sufficiency", fontweight="bold"); ax.legend()
    save(fig, "F7_retrieval_sufficiency")

# ---- F8 risk-coverage curve ----
if len(sel):
    fig, ax = plt.subplots(figsize=(7,4.2))
    ax.plot(cov, acc_at, color=KAU_GREEN, lw=2)
    ax.fill_between(cov, acc_at, alpha=.12, color=KAU_GREEN)
    for t,c in METRICS["metacognition"]["coverage_at_accuracy"].items():
        if c>0: ax.axvline(c, ls=":", c=KAU_GOLD); ax.text(c, .35, f" acc≥{t}", rotation=90, fontsize=8)
    ax.set_xlabel("coverage (fraction retained)"); ax.set_ylabel("accuracy on retained")
    ax.set_title(f"Selective prediction · AUARC = {auarc:.3f}", fontweight="bold")
    save(fig, "F8_risk_coverage")

# ---- F9 reliability ----
fig, ax = plt.subplots(figsize=(6.6,4))
a = METRICS["reliability"]["alpha_intra_run"]; lo,hi = METRICS["reliability"]["alpha_ci95"]
ax.barh(["Krippendorff α\n(intra-run)"], [a], color=KAU_GREEN, height=.45)
ax.errorbar([a],[0], xerr=[[a-lo],[hi-a]], fmt="none", ecolor="#333", capsize=4)
for t,lab in [(0.667,"tentative"),(0.80,"reliable")]:
    ax.axvline(t, ls="--", c="grey", lw=1); ax.text(t, .28, lab, fontsize=8, rotation=90)
ax.set_xlim(0,1); ax.set_title("Label reliability across repeated runs", fontweight="bold")
save(fig, "F9_reliability")

# ---- F10 labeling-function coverage / conflict ----
if len(lf_stats_df):
    fig, ax = plt.subplots(figsize=(8,3.6))
    x = np.arange(len(lf_stats_df)); w=.26
    ax.bar(x-w, lf_stats_df.coverage, w, label="coverage", color=KAU_GREEN)
    ax.bar(x,   lf_stats_df.overlap,  w, label="overlap",  color=KAU_GOLD)
    ax.bar(x+w, lf_stats_df.conflict, w, label="conflict", color="#A44A3F")
    ax.set_xticks(x); ax.set_xticklabels(lf_stats_df.source, rotation=20, ha="right", fontsize=8)
    ax.set_title("Distant-supervision labeling functions", fontweight="bold"); ax.legend()
    save(fig, "F10_labeling_functions")

# ---- F11 extraction quality ----
q = QUALITY.df()
if len(q):
    fig, ax = plt.subplots(figsize=(7.5,3.8))
    d = q.sort_values("align_rate")
    ax.barh(d.role + " · " + d.file.str.slice(0,22), d.align_rate, color=KAU_GREEN)
    ax.axvline(0.90, ls="--", c=KAU_GOLD); ax.set_xlim(0,1)
    ax.set_title("Arabic dual-source merge — token alignment rate", fontweight="bold")
    save(fig, "F11_extraction_quality")

# ---- F12 baseline vs system ----
fig, ax = plt.subplots(figsize=(6.4,4))
ax.bar(["embedding\nbaseline","multi-agent\nsystem"],
       [METRICS["baseline"]["baseline_mutation_accuracy"],
        METRICS["baseline"]["system_mutation_accuracy"]], color=["#7D8CA3", KAU_GREEN])
ax.set_ylim(0,1); ax.set_ylabel("mutation recovery accuracy")
ax.set_title(f"Architecture margin: {METRICS['baseline']['margin_pp']} pp", fontweight="bold")
save(fig, "F12_baseline_margin")
print("\nfigures written to", FIG)

### 17.1 · Architecture diagrams

In [ ]:
import matplotlib.patches as mp

def box(ax, x, y, w, h, label, sub="", fc="#EAF2ED", ec=KAU_GREEN, fs=9):
    ax.add_patch(mp.FancyBboxPatch((x,y), w, h, boxstyle="round,pad=0.012",
                                   fc=fc, ec=ec, lw=1.5))
    ax.text(x+w/2, y+h*0.62, label, ha="center", va="center", fontsize=fs, fontweight="bold")
    if sub: ax.text(x+w/2, y+h*0.26, sub, ha="center", va="center", fontsize=fs-2, color="#444")

def arrow(ax, x1, y1, x2, y2, txt="", c=KAU_GOLD, style="-|>"):
    ax.annotate("", xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle=style, lw=1.6, color=c))
    if txt: ax.text((x1+x2)/2, (y1+y2)/2+0.012, txt, fontsize=7.5, ha="center", color="#333")

# ---- D1 system architecture ----
fig, ax = plt.subplots(figsize=(12,6.2)); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
ax.text(.5,.965,"Metacognitive Multi-Agent Gap-Analysis Pipeline", ha="center",
        fontsize=14, fontweight="bold", color=KAU_GREEN)
box(ax,.03,.72,.17,.14,"GDPR corpus","articles + recitals", fc="#FFF7E6", ec=KAU_GOLD)
box(ax,.03,.52,.17,.14,"Requirement units","frozen instrument, 8-slot frame", fc="#FFF7E6", ec=KAU_GOLD)
box(ax,.03,.30,.17,.14,"Target statutes","dual-source Arabic merge", fc="#FFF7E6", ec=KAU_GOLD)
box(ax,.25,.42,.15,.16,"Retriever","bilingual, top-k")
box(ax,.45,.42,.15,.16,"Solver","gpt-5 · frame → slots → verdict")
box(ax,.66,.42,.15,.16,"Critic","Llama-3.3-70B · falsify")
box(ax,.45,.70,.36,.14,"Orchestrator — metacognitive monitoring & control",
    "sufficiency · strategy · completeness · reflection", fc="#E7EFF3", ec="#4E8098")
box(ax,.86,.42,.11,.16,"Verdict rule","deterministic", fc="#EDEDED", ec="#666")
box(ax,.25,.14,.72,.13,"Records + full trace  →  evaluation suite",
    "mutation · negative controls · distant supervision · MR · α · ablation", fc="#F3F0E4", ec=KAU_GOLD)
arrow(ax,.20,.59,.25,.53); arrow(ax,.20,.37,.25,.47)
arrow(ax,.40,.50,.45,.50,"candidates")
arrow(ax,.60,.50,.66,.50,"record")
arrow(ax,.66,.46,.60,.46,"objection", c="#A44A3F", style="-|>")
arrow(ax,.63,.70,.55,.58); arrow(ax,.72,.70,.73,.58); arrow(ax,.50,.70,.33,.58)
arrow(ax,.81,.50,.86,.50)
arrow(ax,.91,.42,.91,.27); arrow(ax,.52,.42,.52,.27)
ax.text(.83,.365,"bounded retries\nwith structured objection", fontsize=7, ha="center", color="#A44A3F")
save(fig,"D1_architecture")

# ---- D2 evaluation protocol map ----
fig, ax = plt.subplots(figsize=(12,5.6)); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
ax.text(.5,.95,"Evaluation Protocol — what each method establishes", ha="center",
        fontsize=14, fontweight="bold", color=KAU_GREEN)
groups = [
 (.02,"VALIDITY","#EAF2ED",KAU_GREEN,
  ["Mutation testing\n(synthetic ground truth)","Negative controls\n(distractor injection)",
   "Distant supervision\n(reference KB → LFs)"]),
 (.27,"RELIABILITY","#FFF7E6",KAU_GOLD,
  ["Krippendorff α\nintra-run","α inter-prompt","α inter-backbone"]),
 (.52,"ROBUSTNESS","#E7EFF3","#4E8098",
  ["Invariant MRs","Directional MRs","Grounding &\nfaithfulness"]),
 (.77,"CAUSALITY","#F7ECEA","#A44A3F",
  ["Ablation grid\n(5 configurations)","Baseline comparison","Metacognitive\nreadouts"]),
]
for x, title, fc, ec, items in groups:
    ax.add_patch(mp.FancyBboxPatch((x,.10),.21,.74, boxstyle="round,pad=0.012",
                                   fc=fc, ec=ec, lw=1.6, alpha=.55))
    ax.text(x+.105,.80,title, ha="center", fontsize=11, fontweight="bold", color=ec)
    for i,it in enumerate(items):
        ax.add_patch(mp.FancyBboxPatch((x+.015,.62-i*.19),.18,.145,
                                       boxstyle="round,pad=0.008", fc="white", ec=ec, lw=1))
        ax.text(x+.105,.6925-i*.19, it, ha="center", va="center", fontsize=8)
ax.text(.5,.04,"No method in this protocol requires annotation by the research team.",
        ha="center", fontsize=9, style="italic", color="#555")
save(fig,"D2_evaluation_map")

# ---- D3 leakage-control diagram ----
fig, ax = plt.subplots(figsize=(11,4.4)); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
ax.text(.5,.93,"Leakage Control", ha="center", fontsize=13, fontweight="bold", color=KAU_GREEN)
box(ax,.03,.55,.24,.22,"Reference knowledge base","2 comparison PDFs", fc="#F7ECEA", ec="#A44A3F")
box(ax,.03,.20,.24,.22,"Ground-truth artefacts","mutations · neg. controls", fc="#F7ECEA", ec="#A44A3F")
box(ax,.36,.38,.2,.24,"LeakageGuard","8-gram fingerprint index\n+ secret registry",
    fc="#EDEDED", ec="#333")
box(ax,.65,.55,.3,.22,"Agent path","Retriever · Solver · Critic")
box(ax,.65,.20,.3,.22,"Evaluation path","metrics only", fc="#EAF2ED", ec=KAU_GREEN)
arrow(ax,.27,.66,.36,.56,"register",c="#A44A3F"); arrow(ax,.27,.31,.36,.44,"register",c="#A44A3F")
arrow(ax,.56,.52,.65,.62,"scan every prompt")
arrow(ax,.27,.25,.65,.28,"direct (permitted)",c=KAU_GREEN)
ax.text(.605,.44,"✗ raises\nLeakageError", fontsize=8, color="#A44A3F", ha="center", fontweight="bold")
save(fig,"D3_leakage_control")

## 18 · Report and manifest

In [ ]:
# ---- consolidated metrics ----
METRICS["run"] = dict(run_tag=RUN_TAG, config_hash=CONFIG_HASH, seed=SEED,
                      finished=time.strftime("%Y-%m-%d %H:%M:%S"),
                      n_requirement_units=len(UNITS), n_jurisdictions=len(STATUTE_JURIS),
                      n_main_records=int(main_df.label.notna().sum()),
                      n_stability_records=int(STAB.n), n_mutations=int(len(mut_df)),
                      n_negative_controls=int(len(neg_df)), n_mr=int(len(mr_df)),
                      n_ablation=int(len(abl_df)+len(ablmut_df)),
                      total_llm_calls=int(TRACE.n),
                      guard_checks=int(GUARD.checks),
                      guard_fingerprints=int(len(GUARD.fingerprints)),
                      leakage_violations=0)

def _clean(o):
    if isinstance(o, dict):  return {k:_clean(v) for k,v in o.items()}
    if isinstance(o, list):  return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return (None if np.isnan(o) else float(o))
    if isinstance(o, float) and np.isnan(o): return None
    return o

(DIRS["09_metrics"]/"metrics.json").write_text(
    json.dumps(_clean(METRICS), indent=2, ensure_ascii=False), encoding="utf-8")

# tidy CSV exports
main_df.to_csv(DIRS["06_records"]/"main_records.csv", index=False)
pivot.to_csv(DIRS["09_metrics"]/"gap_profile_counts.csv")
pct.round(2).to_csv(DIRS["09_metrics"]/"gap_profile_pct.csv")
per_label_df.to_csv(DIRS["09_metrics"]/"mutation_per_label.csv", index=False)
mr_res_df.to_csv(DIRS["09_metrics"]/"metamorphic.csv", index=False)
abl_res.to_csv(DIRS["09_metrics"]/"ablation.csv", index=False)
lf_stats_df.to_csv(DIRS["09_metrics"]/"labeling_functions.csv", index=False)
QUALITY.df().to_csv(DIRS["09_metrics"]/"extraction_quality.csv", index=False)
print("metrics + CSVs written")

In [ ]:
# ---- markdown report ----
def f(x, n=3):
    try:
        if x is None or (isinstance(x,float) and np.isnan(x)): return "n/a"
        return f"{x:.{n}f}"
    except Exception: return str(x)

M = METRICS
rep = f"""# Legal Gap Analysis — GCC vs GDPR · Run `{RUN_TAG}`

**config hash** `{CONFIG_HASH}` · **seed** {SEED} · **finished** {M['run']['finished']}
Solver `{CONFIG['solver_model']}` · Critic `{CONFIG['critic_model']}`

## 1 Scale
| | |
|---|---|
| GDPR requirement units | {M['run']['n_requirement_units']} |
| Jurisdictions | {M['run']['n_jurisdictions']} ({', '.join(STATUTE_JURIS)}) |
| Main records | {M['run']['n_main_records']} |
| Stability replicates | {M['run']['n_stability_records']} (k={CONFIG['stability_runs']}) |
| Mutations | {M['run']['n_mutations']} |
| Negative controls | {M['run']['n_negative_controls']} |
| Metamorphic tests | {M['run']['n_mr']} |
| Ablation records | {M['run']['n_ablation']} |
| Total LLM calls | {M['run']['total_llm_calls']} |

## 2 Leakage control
Guard checked **{M['run']['guard_checks']:,}** prompts against **{M['run']['guard_fingerprints']:,}**
reference n-gram fingerprints and {len(GUARD.secrets)} registered ground-truth secrets.
Self-test: primary text passes = {GUARD_SELFTEST['primary_passes']}, reference prose blocked = {GUARD_SELFTEST['reference_blocked']}. Violations: **{M['run']['leakage_violations']}** (a violation raises, so a completed run implies zero).

## 3 Validity
| Metric | Value | 95% CI |
|---|---|---|
| Mutation recovery accuracy | {f(M['mutation_recovery']['accuracy'])} | [{f(M['mutation_recovery']['ci95'][0])}, {f(M['mutation_recovery']['ci95'][1])}] |
| Hallucinated coverage (negative controls) | {f(M['safety']['hallucinated_coverage'])} | [{f(M['safety']['hallucinated_ci95'][0])}, {f(M['safety']['hallucinated_ci95'][1])}] |
| **False-C5 rate** (hallucinated absence) | **{f(M['safety']['false_C5_rate'])}** | [{f(M['safety']['false_C5_ci95'][0])}, {f(M['safety']['false_C5_ci95'][1])}] |
| Agreement with distant-supervision weak labels | {f(M['distant_supervision']['agreement'])} | [{f(M['distant_supervision']['ci95'][0])}, {f(M['distant_supervision']['ci95'][1])}] |

Distant-supervision coverage: **{M['distant_supervision']['covered_cells']}/{M['distant_supervision']['total_cells']}**
cells ({f(M['distant_supervision']['coverage_frac'],2)}). Recall is reported on this denominator only;
incomplete knowledge-base coverage is a known noise source and weak labels bound rather than measure performance.

## 4 Reliability
Krippendorff α (intra-run, k={CONFIG['stability_runs']}, custom non-ordinal difference function):
**{f(M['reliability']['alpha_intra_run'])}** 95% CI [{f(M['reliability']['alpha_ci95'][0])}, {f(M['reliability']['alpha_ci95'][1])}]
Mean raw agreement {f(M['reliability']['raw_agreement'])} · max label prevalence {f(M['reliability']['max_prevalence'])}.
Reliability is reported as a precondition for validity, never as evidence of correctness.

## 5 Robustness and grounding
| | |
|---|---|
| Citation id valid | {f(M['grounding']['citation_id_valid'])} |
| Citation span verified in source | {f(M['grounding']['citation_span_valid'])} |
| Modality claims supported by a deontic operator | {f(M['grounding']['deontic_supported'])} |
| Rationale faithfulness (rule vs holistic label) | {f(M['grounding']['rationale_faithfulness'])} |
| Mean slot completeness | {f(M['grounding']['mean_slot_completeness'])} |
| C5 records carrying an exclusion list | {f(M['grounding']['c5_with_exclusion_list'])} |
| Cross-jurisdiction contradictions | {M['consistency']['cross_jurisdiction_contradictions']} ({f(M['consistency']['cross_jurisdiction_rate'],4)}) |
| Incoherent C1 (unfilled mandatory slot) | {M['consistency']['inter_slot_incoherent_C1']} |

## 6 Architecture justification
Embedding baseline {f(M['baseline']['baseline_mutation_accuracy'])} vs system
{f(M['baseline']['system_mutation_accuracy'])} — margin **{M['baseline']['margin_pp']} pp**.
Adjusted Rand between baseline and system labels: {f(M['baseline']['adjusted_rand'])}
(low ARI with a high margin is the desired pattern: the architecture is not reproducing a similarity threshold).

## 7 Metacognition
| | |
|---|---|
| Trigger precision / recall | {f(M['metacognition']['trigger_precision'])} / {f(M['metacognition']['trigger_recall'])} |
| AUARC (selective prediction) | {f(M['metacognition']['AUARC'])} |
| Type-2 AUROC (metacognitive sensitivity) | {f(M['metacognition']['type2_AUROC'])} |
| Overthinking rate | {f(M['metacognition']['overthinking_rate'])} |
| Coverage achievable at accuracy targets | {M['metacognition']['coverage_at_accuracy']} |

## 8 Limitations
- Verdicts not covered by mutation, negative controls or the reference knowledge base are
  **candidate findings requiring legal validation**, not validated results.
- Distant supervision provides bounds, not an unbiased accuracy estimate.
- Textual scope only: no claim about enforcement, regulatory practice or adequacy.
- Corpus frozen at the ingestion date recorded in `00_manifest/manifest.json`.

---
Figures: `10_figures/` · Metrics: `09_metrics/metrics.json` · Full trace: `05_traces/llm_calls.jsonl`
"""
(DIRS["11_report"]/"REPORT.md").write_text(rep, encoding="utf-8")
MANIFEST["finished"] = M["run"]["finished"]; MANIFEST["metrics_summary"] = _clean(M["run"])
(DIRS["00_manifest"]/"manifest.json").write_text(
    json.dumps(MANIFEST, indent=2, ensure_ascii=False), encoding="utf-8")

from IPython.display import Markdown, display as disp
disp(Markdown(rep))
print("\n" + "="*70)
print("Report  →", DIRS["11_report"]/"REPORT.md")
print("Figures →", FIG)
print("All outputs under:", OUT)